<a href="https://colab.research.google.com/github/robertbarcik/genai-in-python-tutorial/blob/main/2_basic_project_examples/5_extracting_information_from_unstructured_data/5_extracting_information_from_unstructured_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Extracting Structured Information from Unstructured Text

By the end of this notebook, you'll be able to:

1. **Extract structured data** from messy, unstructured text (emails, tickets, reports)
2. **Choose the right format** - CSV, JSON, or Pydantic models for your use case
3. **Parse complex information** - Handle nested data, arrays, and multiple items
4. **Validate extractions** - Ensure data quality and catch errors early
5. **Save results** - Store extracted data in files for downstream use
6. **Handle edge cases** - Deal with missing info, contradictions, and ambiguity
7. **Build production systems** - Create robust, scalable extraction pipelines

---

## The Business Problem

In IT support and services, information constantly flows in **unstructured formats** - free-form text that humans can read but computers struggle to process:

- **Emails**: "Hi, my laptop won't start. I think it's the battery. Can someone help? - John from Marketing"
- **Chat messages**: "printer broken room 304 need toner asap"
- **Verbal reports**: "Sarah mentioned something about the network being slow in Building B"
- **Handwritten notes**: Notes from a phone call or site visit

But to **take action**, this information needs to be **structured** - organized into consistent fields that systems can store, query, and analyze:

```json
{
  "user_name": "John",
  "department": "Marketing",
  "issue": "Laptop won't start",
  "suspected_cause": "Battery",
  "urgency": "medium"
}
```

### Current Challenges

Traditionally, converting unstructured data to structured data requires manual data entry, which is:

- **Slow** - Takes time away from actual problem-solving
- **Error-prone** - Typos, missed fields, inconsistent formatting
- **Doesn't scale** - Can't handle high ticket volumes
- **Inconsistent** - Different people extract different information

### The Solution: LLM-Powered Extraction

Large Language Models can automate this entire process. Given unstructured text, they can:

- Extract key information accurately and consistently
- Handle natural language variations ("urgent", "asap", "critical")
- Infer missing information from context
- Structure data in your required format (JSON, CSV, database schema)
- Process hundreds of items in minutes

---

## Key Concepts

Before we start coding, let's clarify two fundamental concepts that underpin this entire notebook:

**Unstructured Data** is free-form text with no fixed format - emails, chat messages, documents, verbal reports. It's easy for humans to read but hard for computers to process directly.

**Structured Data** is organized in a predefined format - database tables, JSON objects, CSV files. It's easy to query, analyze, and integrate with other systems.

### Data Extraction vs. Data Parsing

These two steps work together to convert unstructured input into usable output:

**Data Extraction** means identifying and pulling out specific information from unstructured text. It requires understanding context and meaning - for example, finding a user's name, issue type, and urgency level from an email.

**Data Parsing** means converting the extracted information into a structured format, such as creating a JSON object with the extracted fields.

LLMs excel at **both** - they understand context AND can output structured formats.

---

# Setup

Let's configure the environment and install required libraries.

## Upload Data Files

This notebook uses external data files for some examples. Upload the following files to your Colab environment before running the notebook:

1. Click the folder icon in the left sidebar
2. Click the upload button
3. Upload these files:
   - `conference_room_inventory.txt`
   - `conference_rooms_detailed.txt`



## Install Dependencies

We'll install five libraries:
- **openai**: Official OpenAI Python client for API access
- **pydantic**: Data validation and settings management using Python type hints
- **email-validator**: Email validation for Pydantic's EmailStr type
- **tqdm**: Progress bars for batch processing
- **pandas**: Data manipulation and CSV handling

In [1]:
!pip install -q openai==2.28.0 pydantic==2.12.3 email-validator==2.3.0 tqdm==4.67.3 pandas==2.2.2

print("✅ All dependencies installed!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.83.4 requires openai==2.30.0, but you have openai 2.28.0 which is incompatible.
litellm 1.83.4 requires pydantic==2.12.5, but you have pydantic 2.12.3 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


✅ All dependencies installed!


## API Key Configuration

You have two methods to provide your API key:

**Method 1 (Recommended)**: Use Colab Secrets
1. Click the 🔑 icon in the left sidebar
2. Click "Add new secret"
3. Name: `OPENAI_API_KEY`
4. Value: Your OpenAI API key
5. Enable notebook access

**Method 2 (Fallback)**: Manual input when prompted

Run the cell below to configure authentication:

In [2]:
import os

# Configure OpenAI API key
# Method 1: Try to get API key from Colab secrets (recommended)
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("✅ API key loaded from Colab secrets")
except:
    import os
    if os.environ.get('OPENAI_API_KEY'):
        OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
        print("✅ API key loaded from environment variable")
    else:
        # Method 2: Manual input (fallback)
        from getpass import getpass
        print("💡 To use Colab secrets: Go to 🔑 (left sidebar) → Add new secret → Name: OPENAI_API_KEY")
        OPENAI_API_KEY = getpass("Enter your OpenAI API Key: ")

# Set the API key as an environment variable
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Validate that the API key is set
if not OPENAI_API_KEY or OPENAI_API_KEY.strip() == "":
    raise ValueError("❌ ERROR: No API key provided!")

print("✅ Authentication configured!")

# Configure which OpenAI model to use
OPENAI_MODEL = "gpt-5-nano"  # Using gpt-5-nano for cost efficiency
print(f"Selected Model: {OPENAI_MODEL}")

✅ API key loaded from environment variable
✅ Authentication configured!
Selected Model: gpt-5-nano


## Import Required Libraries

Now let's import all the libraries we'll use throughout this tutorial:

In [3]:
from openai import OpenAI
import json
import csv
from datetime import datetime
from typing import Optional, List, Dict, Any
from enum import Enum
from pydantic import BaseModel, EmailStr, Field, field_validator
import pandas as pd
from tqdm import tqdm
from IPython.display import display

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## Initialize OpenAI Client

Now let's create a client instance to interact with the OpenAI API:

In [4]:
# Initialize the OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

print("✅ OpenAI client initialized!")

✅ OpenAI client initialized!


---

# Understanding Output Formats

LLMs can extract data into different formats. The format you choose depends on your use case - how complex the data is, where it needs to go, and how strictly it needs to be validated.

We'll explore three main formats, each with increasing complexity and validation:
1. **CSV** - Simple tabular data, great for spreadsheets
2. **JSON** - Complex, nested structures, standard for APIs
3. **Pydantic Models** - Type-safe, validated data for production systems

---

## Format A: CSV (Comma-Separated Values)

### When to Use CSV

CSV is the simplest structured format - each row represents one item, and columns represent fields. It's best for **simple tabular data** where all items have the same flat structure.

**Common use cases in IT:**
- Asset inventories and equipment lists
- Hardware tracking spreadsheets
- Simple databases that need Excel compatibility
- Reports for non-technical stakeholders

**Pros:** Easy to open in Excel or Google Sheets, simple structure, human-readable and editable.

**Cons:** No nesting (can't represent complex relationships), all values are strings, harder to represent one-to-many relationships.

### Practical Example: Conference Room Inventory

In this example, we'll load an unstructured text description of conference room equipment and ask the LLM to extract it into a clean CSV table. The input file contains free-form sentences describing equipment in three rooms - the kind of data you might find in a facility report or email.

In [5]:
# Load unstructured inventory description from file
with open("conference_room_inventory.txt", "r") as f:
    inventory_description = f.read()

print("Loaded inventory description:")
print(inventory_description)

# Extraction prompt for CSV format
extraction_prompt = f"""
Extract equipment inventory information from the text below and format it as CSV.

Use these exact column headers: room,device_type,manufacturer,model,serial_number

Rules:
- One row per device
- Include header row
- Use commas as separators
- No quotes around values unless they contain commas

Text:
{inventory_description}
"""

print("\n🔄 Extracting inventory data to CSV format...\n")

# Make API call using Responses API
response = client.responses.create(
    model=OPENAI_MODEL,
    input=extraction_prompt,
    text={"verbosity": "low"}  # Low verbosity for structured output
)

csv_output = response.output_text.strip()

print("Extracted CSV Data:")
print("=" * 80)
print(csv_output)
print("=" * 80)

# Save to file
csv_file_path = "./inventory_data.csv"
with open(csv_file_path, 'w') as f:
    f.write(csv_output)

print(f"\nSaved to: {csv_file_path}")
print(f"Tokens used: {response.usage.total_tokens}")

Loaded inventory description:
Conference Room A has a Samsung 65-inch display (serial: SAMS-2024-001),
a Logitech Rally camera (serial: LOG-CAM-445), and a Poly Studio phone system (serial: POLY-899-X).

Conference Room B contains two Dell OptiPlex 7090 computers (serials: DELL-PC-1023 and DELL-PC-1024),
an LG 55-inch display (serial: LG-DSP-3301), and a Jabra Speak 750 speakerphone (serial: JAB-750-229).

Conference Room C is equipped with a Microsoft Surface Hub 2S (serial: MSFT-HUB-8821),
a Cisco Webex Room Kit (serial: CISCO-WX-4492), and an HP laptop (serial: HP-LT-9933).


🔄 Extracting inventory data to CSV format...



Extracted CSV Data:
room,device_type,manufacturer,model,serial_number
Conference Room A,display,Samsung,65-inch display,SAMS-2024-001
Conference Room A,camera,Logitech,Rally,LOG-CAM-445
Conference Room A,phone system,Poly,Studio,POLY-899-X
Conference Room B,computer,Dell,OptiPlex 7090,DELL-PC-1023
Conference Room B,computer,Dell,OptiPlex 7090,DELL-PC-1024
Conference Room B,display,LG,55-inch display,LG-DSP-3301
Conference Room B,speakerphone,Jabra,Speak 750,JAB-750-229
Conference Room C,computer,Microsoft,Surface Hub 2S,MSFT-HUB-8821
Conference Room C,video system,Cisco,Room Kit,CISCO-WX-4492
Conference Room C,computer,HP,laptop,HP-LT-9933

Saved to: ./inventory_data.csv
Tokens used: 3043


In [6]:
# Verify the CSV is valid by parsing it back
print("🔍 Verifying CSV format...\n")

# Method 1: Using pandas
df = pd.read_csv(csv_file_path)
print("✅ CSV is valid and readable!\n")
print(f" Found {len(df)} devices across {df['room'].nunique()} rooms\n")
print("Preview:")

# Display as DataFrame (better formatting in Jupyter)
display(df)

🔍 Verifying CSV format...

✅ CSV is valid and readable!

 Found 10 devices across 3 rooms

Preview:


,room,device_type,manufacturer,model,serial_number
0,Conference Room A,display,Samsung,65-inch display,SAMS-2024-001
1,Conference Room A,camera,Logitech,Rally,LOG-CAM-445
2,Conference Room A,phone system,Poly,Studio,POLY-899-X
3,Conference Room B,computer,Dell,OptiPlex 7090,DELL-PC-1023
4,Conference Room B,computer,Dell,OptiPlex 7090,DELL-PC-1024
5,Conference Room B,display,LG,55-inch display,LG-DSP-3301
6,Conference Room B,speakerphone,Jabra,Speak 750,JAB-750-229
7,Conference Room C,computer,Microsoft,Surface Hub 2S,MSFT-HUB-8821
8,Conference Room C,video system,Cisco,Room Kit,CISCO-WX-4492
9,Conference Room C,computer,HP,laptop,HP-LT-9933


---

## Format B: JSON (JavaScript Object Notation)

### When to Use JSON

JSON is a more flexible format than CSV - it supports nested objects, arrays, and multiple data types. This makes it the preferred choice when your data has complex relationships or varying structure.

**Common use cases in IT:**
- Support tickets with multiple fields and categories
- User information with device specifications
- Error reports with nested details
- API integration and data exchange
- Configuration files and settings

**Pros:** Flexible structure with nesting, native data types (strings, numbers, booleans, arrays), standard format for APIs and modern applications.

**Cons:** More verbose than CSV, less human-friendly for simple tables.

### Practical Example: Support Ticket Extraction

In this example, we'll extract structured information from a support ticket email. Unlike the CSV example, this data has fields of different types (strings, lists) that benefit from JSON's flexibility.

In [7]:
# Unstructured support ticket email
support_ticket = """
From: jennifer.martinez@company.com
Subject: Urgent - Cannot Access Email

Hi IT Support,

I'm Jennifer Martinez from the Sales department (employee ID: EMP-5834).
My Outlook keeps crashing whenever I try to open it. I've tried restarting
my computer twice but the problem persists.

This is really urgent because I need to send quotes to clients today.
My phone number is 555-0192 if you need to call me.

Please help ASAP!

Thanks,
Jennifer
"""

print("Support ticket email loaded:")
print(support_ticket)

Support ticket email loaded:

From: jennifer.martinez@company.com
Subject: Urgent - Cannot Access Email

Hi IT Support,

I'm Jennifer Martinez from the Sales department (employee ID: EMP-5834).
My Outlook keeps crashing whenever I try to open it. I've tried restarting
my computer twice but the problem persists.

This is really urgent because I need to send quotes to clients today.
My phone number is 555-0192 if you need to call me.

Please help ASAP!

Thanks,
Jennifer



In [8]:
# Extraction prompt for JSON format
extraction_prompt = f"""
Extract support ticket information from the email below and format it as JSON.

Include these fields:
- user_name: Full name of the user
- department: User's department
- employee_id: Employee ID if mentioned
- contact_email: Email address
- contact_phone: Phone number if mentioned
- issue_summary: Brief summary of the issue
- application: The application having issues
- urgency: Low, Medium, High, or Critical (infer from context)
- actions_tried: List of troubleshooting steps user already attempted

Email:
{support_ticket}

Respond with ONLY valid JSON, no additional text.
"""

print("🔄 Extracting ticket data to JSON format...\n")

# Make API call
response = client.responses.create(
    model=OPENAI_MODEL,
    input=extraction_prompt,
    text={"verbosity": "low"}
)

json_output = response.output_text.strip()

# Parse and pretty-print the JSON
ticket_data = json.loads(json_output)

print("Extracted JSON Data:")
print("=" * 80)
print(json.dumps(ticket_data, indent=2))
print("=" * 80)

# Save to file
json_file_path = "./ticket_data.json"
with open(json_file_path, 'w') as f:
    json.dump(ticket_data, f, indent=2)

print(f"\nSaved to: {json_file_path}")
print(f"Tokens used: {response.usage.total_tokens}")

🔄 Extracting ticket data to JSON format...



Extracted JSON Data:
{
  "user_name": "Jennifer Martinez",
  "department": "Sales",
  "employee_id": "EMP-5834",
  "contact_email": "jennifer.martinez@company.com",
  "contact_phone": "555-0192",
  "issue_summary": "Outlook keeps crashing on open; unable to access email",
  "application": "Microsoft Outlook",
  "urgency": "High",
  "actions_tried": [
    "Restarted computer twice"
  ]
}

Saved to: ./ticket_data.json
Tokens used: 939


In [9]:
# Verify and demonstrate JSON parsing
print("🔍 Verifying JSON format and demonstrating access...\n")

# Read back from file
with open(json_file_path, 'r') as f:
    loaded_data = json.load(f)

print("✅ JSON is valid!\n")
print("Accessing specific fields:")
print(f"  User: {loaded_data['user_name']}")
print(f"  Department: {loaded_data['department']}")
print(f"  Issue: {loaded_data['issue_summary']}")
print(f"  Urgency: {loaded_data['urgency']}")
print(f"  Actions tried: {', '.join(loaded_data['actions_tried'])}")

🔍 Verifying JSON format and demonstrating access...

✅ JSON is valid!

Accessing specific fields:
  User: Jennifer Martinez
  Department: Sales
  Issue: Outlook keeps crashing on open; unable to access email
  Urgency: High
  Actions tried: Restarted computer twice


---

## Format C: Pydantic Models (Type-Safe Validation)

### When to Use Pydantic

Pydantic is a Python library that validates data against a defined schema using type hints. While JSON extraction gives you structured data, it doesn't guarantee that the data is *correct* - an email field might be missing the @ sign, an employee ID might be in the wrong format, or a required field might be empty.

Pydantic catches these problems automatically at the moment you try to create the object. This is essential for production systems where bad data could corrupt databases or cause downstream errors.

### Comparison: JSON vs. Pydantic

Let's see the difference between plain JSON (no validation) and Pydantic (validated):

#### Problem: Plain JSON Without Validation

In [10]:
# Example: Plain JSON - problems discovered later
bad_ticket = {
    "user_name": "John",  # Missing last name
    "employee_id": "12345",  # Wrong format (should be EMP-####)
    "contact_email": "john.company.com",  # Invalid email (missing @)
    "urgency": "super urgent",  # Invalid value (should be Low/Medium/High/Critical)
    # Missing required field: issue_summary
}

print("❌ Plain JSON - No validation happens:")
print(json.dumps(bad_ticket, indent=2))
print("\n⚠️ This bad data could be saved to database, causing errors later!")

❌ Plain JSON - No validation happens:
{
  "user_name": "John",
  "employee_id": "12345",
  "contact_email": "john.company.com",
  "urgency": "super urgent"
}

⚠️ This bad data could be saved to database, causing errors later!


#### Solution: Pydantic Model with Validation

Let's create a Pydantic model that validates our data:

In [11]:
# Define urgency levels as an Enum
class UrgencyLevel(str, Enum):
    LOW = "Low"
    MEDIUM = "Medium"
    HIGH = "High"
    CRITICAL = "Critical"

# Define the Pydantic model for a support ticket
class SupportTicket(BaseModel):
    # Required fields
    user_name: str = Field(..., min_length=2, description="Full name of the user")
    employee_id: str = Field(..., pattern=r'^EMP-\d{4}$', description="Employee ID in format EMP-####")
    contact_email: EmailStr = Field(..., description="Valid email address")
    issue_summary: str = Field(..., min_length=10, description="Brief summary of the issue")
    urgency: UrgencyLevel = Field(..., description="Urgency level")

    # Optional fields with defaults
    department: Optional[str] = None
    contact_phone: Optional[str] = None
    application: Optional[str] = None
    actions_tried: List[str] = Field(default_factory=list)

    # Custom validator for phone numbers
    @field_validator('contact_phone')
    @classmethod
    def validate_phone(cls, v):
        if v and len(v.replace('-', '').replace(' ', '')) < 7:
            raise ValueError('Phone number too short')
        return v

print("✅ Pydantic model defined with validation rules!")

✅ Pydantic model defined with validation rules!


#### Test 1: Invalid Data (Validation Catches Errors)

In [12]:
# Try to create ticket with bad data
print("Testing with INVALID data...\n")

try:
    bad_ticket_validated = SupportTicket(
        user_name="J",  # Too short
        employee_id="12345",  # Wrong format
        contact_email="invalid-email",  # Invalid email
        issue_summary="Broken",  # Too short
        urgency="super urgent"  # Invalid urgency level
    )
except Exception as e:
    print("❌ Validation failed (as expected):")
    print(f"\nError type: {type(e).__name__}")
    print(f"\nErrors found:")

    # Parse validation errors
    if hasattr(e, 'errors'):
        for error in e.errors():
            field = error['loc'][0]
            message = error['msg']
            print(f"  • {field}: {message}")

    print("\n✅ Validation prevented bad data from being processed!")

Testing with INVALID data...

❌ Validation failed (as expected):

Error type: ValidationError

Errors found:
  • user_name: String should have at least 2 characters
  • employee_id: String should match pattern '^EMP-\d{4}$'
  • contact_email: value is not a valid email address: An email address must have an @-sign.
  • issue_summary: String should have at least 10 characters
  • urgency: Input should be 'Low', 'Medium', 'High' or 'Critical'

✅ Validation prevented bad data from being processed!


#### Test 2: Valid Data (Validation Succeeds)

Now let's use LLM to extract data in Pydantic-compatible format:

In [13]:
# Mock data for extraction
ticket_email = """
From: robert.chen@company.com
Subject: VPN Connection Issues - Need Help

Hello IT,

This is Robert Chen from Engineering (EMP-7821). My VPN client keeps
disconnecting every 5-10 minutes. I've already tried:
- Restarting the VPN client
- Rebooting my laptop
- Checking my internet connection

This is blocking my work as I need to access the development servers.
Please treat this as high priority.

You can reach me at 555-0198.

Thanks,
Robert
"""

Now let's create the extraction prompt and send it to the model:

In [14]:
# Extraction prompt that ensures Pydantic-compatible format
extraction_prompt = f"""
Extract support ticket information and format as JSON matching this schema:

Required fields:
- user_name (string, min 2 chars): Full name
- employee_id (string, format: EMP-####): Employee ID
- contact_email (string): Valid email address
- issue_summary (string, min 10 chars): Brief issue description
- urgency (string): Must be exactly one of: "Low", "Medium", "High", "Critical"

Optional fields:
- department (string or null): Department name
- contact_phone (string or null): Phone number
- application (string or null): Application name
- actions_tried (array of strings): Steps user already tried

Email:
{ticket_email}

Return ONLY valid JSON, no additional text.
"""

print("🔄 Extracting ticket with Pydantic validation...\n")

# Extract data
response = client.responses.create(
    model=OPENAI_MODEL,
    input=extraction_prompt,
    text={"verbosity": "low"}
)

extracted_json = response.output_text.strip()
extracted_data = json.loads(extracted_json)

print(" Extracted JSON:")
print(json.dumps(extracted_data, indent=2))
print()

# Validate with Pydantic
try:
    validated_ticket = SupportTicket(**extracted_data)
    print("✅ Pydantic validation PASSED!\n")
    print("Validated ticket details:")
    print(f"  User: {validated_ticket.user_name}")
    print(f"  Employee ID: {validated_ticket.employee_id}")
    print(f"  Email: {validated_ticket.contact_email}")
    print(f"  Urgency: {validated_ticket.urgency.value}")
    print(f"  Issue: {validated_ticket.issue_summary}")
    print(f"  Actions tried: {len(validated_ticket.actions_tried)} steps")

    # Save validated ticket
    validated_file_path = "./validated_ticket.json"
    with open(validated_file_path, 'w') as f:
        json.dump(validated_ticket.model_dump(), f, indent=2)

    print(f"\n Saved validated ticket to: {validated_file_path}")
    print(f" Tokens used: {response.usage.total_tokens}")

except Exception as e:
    print(f"❌ Validation failed: {e}")

🔄 Extracting ticket with Pydantic validation...



 Extracted JSON:
{
  "user_name": "Robert Chen",
  "employee_id": "EMP-7821",
  "contact_email": "robert.chen@company.com",
  "issue_summary": "VPN client disconnects every 5-10 minutes.",
  "urgency": "High",
  "department": "Engineering",
  "contact_phone": "555-0198",
  "application": "VPN Client",
  "actions_tried": [
    "Restarted the VPN client",
    "Rebooted my laptop",
    "Checked my internet connection"
  ]
}

✅ Pydantic validation PASSED!

Validated ticket details:
  User: Robert Chen
  Employee ID: EMP-7821
  Email: robert.chen@company.com
  Urgency: High
  Issue: VPN client disconnects every 5-10 minutes.
  Actions tried: 3 steps

 Saved validated ticket to: ./validated_ticket.json
 Tokens used: 1454


### Pydantic Benefits Summary

**Pydantic provides:**

✅ **Type safety** - Fields must be correct types  
✅ **Format validation** - Email, patterns, length checks  
✅ **Required field checking** - No missing critical data  
✅ **Enum validation** - Only allowed values accepted  
✅ **Custom validators** - Business logic validation  
✅ **Clear error messages** - Know exactly what's wrong  
✅ **IDE support** - Auto-completion and type hints  

💡 **When to use:** Production systems, database integration, team projects requiring data contracts

---

# Practical Examples

Now let's apply what we've learned to realistic scenarios.

---

## Example 1: Support Ticket Parsing (Comprehensive)

We'll extract information from support tickets, starting with basic extraction and progressing to complex nested structures.

### Part A: Basic Ticket Extraction

We'll start with a straightforward case: extracting standard ticket fields (user info, issue details, urgency) from a support email. The prompt specifies exactly which fields we want, and the LLM returns a flat JSON object with those fields populated from the email content.

In [15]:
# Realistic mock data: User email about laptop issue
laptop_issue_email = """
From: sarah.williams@company.com
Date: 2025-01-15 09:23 AM
Subject: Laptop Battery Problem - Urgent

Hi Support Team,

I'm Sarah Williams from the Marketing department, employee number EMP-4567.
My work laptop (Dell Latitude 5420) isn't holding a charge anymore. The battery
drains completely within an hour even when I'm just using Word and email.

I have a client presentation tomorrow afternoon at 2 PM and really need this
working by then. I've tried using a different power outlet and the charger
seems to work fine (the charging light comes on).

Could someone please help? You can reach me at ext. 4523 or my cell 555-0167.

Thank you!
Sarah Williams
Marketing Department
"""

Now let's create the extraction prompt and send it to the model:

In [16]:
# Extraction prompt
extraction_prompt = f"""
Extract support ticket information from this email and return as JSON.

Include these fields:
- user_name: Full name
- department: Department name
- employee_id: Employee ID
- contact_email: Email address
- contact_phone: Phone number (if mentioned)
- device_info: Device description (manufacturer and model)
- issue_summary: Concise summary of the issue
- issue_details: Detailed description
- urgency: Low, Medium, High, or Critical (infer from context and deadline)
- deadline_context: Any time-sensitive information
- troubleshooting_done: What user already tried

Email:
{laptop_issue_email}

Return ONLY valid JSON.
"""

print("🔄 Extracting basic ticket information...\n")

# Make API call
response = client.responses.create(
    model=OPENAI_MODEL,
    input=extraction_prompt,
    text={"verbosity": "medium"}
)

ticket_json = response.output_text.strip()
ticket_data = json.loads(ticket_json)

print("📋 Extracted Ticket Data:")
print("=" * 80)
print(json.dumps(ticket_data, indent=2))
print("=" * 80)

# Save to file
basic_ticket_path = "./basic_ticket.json"
with open(basic_ticket_path, 'w') as f:
    json.dump(ticket_data, f, indent=2)

print(f"\n Saved to: {basic_ticket_path}")
print(f" Tokens used: {response.usage.total_tokens}")

# Display key insights
print("\n🔍 Key Insights:")
print(f"  • User: {ticket_data['user_name']} ({ticket_data['department']})")
print(f"  • Issue: {ticket_data['issue_summary']}")
print(f"  • Urgency: {ticket_data['urgency']}")
print(f"  • Deadline: {ticket_data['deadline_context']}")

🔄 Extracting basic ticket information...



📋 Extracted Ticket Data:
{
  "user_name": "Sarah Williams",
  "department": "Marketing Department",
  "employee_id": "EMP-4567",
  "contact_email": "sarah.williams@company.com",
  "contact_phone": "ext. 4523; cell 555-0167",
  "device_info": {
    "manufacturer": "Dell",
    "model": "Latitude 5420"
  },
  "issue_summary": "Laptop battery drains quickly; lasts about an hour.",
  "issue_details": "Battery drains completely within an hour even with light use (Word and email). The client presentation is tomorrow at 2 PM, so it is time-sensitive. Tried using a different power outlet; the charger appears to work (charging light is on).",
  "urgency": "High",
  "deadline_context": "Client presentation tomorrow at 2 PM; need laptop functioning by then.",
  "troubleshooting_done": "Tried using a different power outlet; charger seems to work fine (charging light on)."
}

 Saved to: ./basic_ticket.json
 Tokens used: 1657

🔍 Key Insights:
  • User: Sarah Williams (Marketing Department)
  • Issue:

### Part B: Advanced - Nested Device Specifications

Now let's handle more complex data. When a ticket includes detailed hardware specifications (processor, RAM, GPU, storage drives), a flat JSON structure becomes unwieldy. Instead, we'll use **nested objects** - grouping related information together (e.g., all processor details inside a `processor` object). We'll also use **arrays** for multiple items of the same type (e.g., storage drives).

In [17]:
# Complex mock data: Workstation issue with detailed system specs
workstation_issue = """
From: michael.thompson@company.com
Subject: Workstation Performance Issues - Graphics Freezing

Hello IT,

I'm Michael Thompson, CAD Engineer in the Design department (EMP-9012).

My workstation is experiencing severe performance problems. The system specs are:
- Dell Precision 7920 Tower (Service Tag: 5XYZ789)
- Intel Xeon Gold 6248R processor, 24 cores, running at 3.0 GHz
- 128GB DDR4 RAM, ECC memory
- NVIDIA Quadro RTX 5000 with 16GB GDDR6
- Two storage drives: 1TB NVMe SSD (OS drive) and 4TB SATA SSD (data drive)
- Running Windows 11 Pro for Workstations, version 23H2

The screen freezes when I'm rendering 3D models in SolidWorks. Sometimes the
entire application crashes. I suspect it might be the graphics card overheating
because I hear the fans going crazy.

This is blocking a project deadline on Friday. Please help!

Phone: 555-0184
Email: michael.thompson@company.com

Thanks,
Michael
"""

Now let's create the extraction prompt and send it to the model:

In [18]:
# Extraction prompt for nested structure - simplified and clearer
extraction_prompt = f"""
Extract support ticket information with nested device specifications from the email below.

IMPORTANT: Return ONLY valid JSON with NO extra text before or after. Use this exact structure:

{{
  "user_name": "full name",
  "department": "department name",
  "employee_id": "employee ID",
  "contact_email": "email address",
  "contact_phone": "phone number",
  "issue_summary": "brief issue summary",
  "suspected_cause": "suspected cause if mentioned",
  "urgency": "Low or Medium or High or Critical",
  "affected_application": "application name",
  "device": {{
    "manufacturer": "device manufacturer",
    "model": "device model",
    "service_tag": "service tag",
    "processor": {{
      "brand": "processor brand",
      "model": "processor model",
      "cores": 24,
      "speed_ghz": 3.0
    }},
    "ram": {{
      "capacity_gb": 128,
      "type": "DDR4 ECC"
    }},
    "gpu": {{
      "manufacturer": "GPU manufacturer",
      "model": "GPU model",
      "memory_gb": 16
    }},
    "storage": [
      {{
        "capacity_tb": 1.0,
        "type": "NVMe SSD",
        "purpose": "OS drive"
      }},
      {{
        "capacity_tb": 4.0,
        "type": "SATA SSD",
        "purpose": "data drive"
      }}
    ],
    "operating_system": {{
      "name": "Windows 11 Pro for Workstations",
      "version": "23H2"
    }}
  }}
}}

Email to extract from:
{workstation_issue}

Return ONLY the JSON object with no additional text.
"""

print("🔄 Extracting complex nested ticket information...")

# Make API call with lower verbosity for cleaner JSON output
response = client.responses.create(
    model=OPENAI_MODEL,
    input=extraction_prompt,
    text={"verbosity": "low"}  # Low verbosity for cleaner structured output
)

complex_json = response.output_text.strip()

# Try to parse JSON with error handling
try:
    complex_data = json.loads(complex_json)

    print(" Extracted Complex Nested Data:")
    print("=" * 80)
    print(json.dumps(complex_data, indent=2))
    print("=" * 80)

    # Save to file
    complex_ticket_path = "./complex_ticket_nested.json"
    with open(complex_ticket_path, 'w') as f:
        json.dump(complex_data, f, indent=2)

    print(f" Saved to: {complex_ticket_path}")
    print(f" Tokens used: {response.usage.total_tokens}")

    # Demonstrate accessing nested data
    print(" 🔍 Accessing Nested Data:")
    print(f"  • User: {complex_data['user_name']}")
    print(f"  • Device: {complex_data['device']['manufacturer']} {complex_data['device']['model']}")
    print(f"  • CPU: {complex_data['device']['processor']['brand']} {complex_data['device']['processor']['model']}")
    print(f"  • RAM: {complex_data['device']['ram']['capacity_gb']}GB {complex_data['device']['ram']['type']}")
    print(f"  • GPU: {complex_data['device']['gpu']['manufacturer']} {complex_data['device']['gpu']['model']}")
    print(f"  • Storage drives: {len(complex_data['device']['storage'])}")
    for i, drive in enumerate(complex_data['device']['storage'], 1):
        print(f"    - Drive {i}: {drive['capacity_tb']}TB {drive['type']} ({drive['purpose']})")

except json.JSONDecodeError as e:
    print(f"❌ JSON Parsing Error: {e}")
    print(f" Raw response from API:")
    print("=" * 80)
    print(complex_json)
    print("=" * 80)
    print(f"💡 Tip: The model returned invalid JSON. Try adjusting the prompt or using a different model.")

🔄 Extracting complex nested ticket information...


 Extracted Complex Nested Data:
{
  "user_name": "Michael Thompson",
  "department": "Design",
  "employee_id": "EMP-9012",
  "contact_email": "michael.thompson@company.com",
  "contact_phone": "555-0184",
  "issue_summary": "Graphics freezing and occasional crashes during SolidWorks rendering, impacting workflow.",
  "suspected_cause": "Graphics card overheating",
  "urgency": "High",
  "affected_application": "SolidWorks",
  "device": {
    "manufacturer": "Dell",
    "model": "Precision 7920 Tower",
    "service_tag": "5XYZ789",
    "processor": {
      "brand": "Intel",
      "model": "Xeon Gold 6248R",
      "cores": 24,
      "speed_ghz": 3.0
    },
    "ram": {
      "capacity_gb": 128,
      "type": "DDR4 ECC"
    },
    "gpu": {
      "manufacturer": "NVIDIA",
      "model": "Quadro RTX 5000",
      "memory_gb": 16
    },
    "storage": [
      {
        "capacity_tb": 1.0,
        "type": "NVMe SSD",
        "purpose": "OS drive"
      },
      {
        "capacity_tb": 4.0,
 

### Key Points

From Example 1, we learned:

**Flat vs. Nested Structures:**
- ✅ Flat structure: Simple tickets with top-level fields only
- ✅ Nested structure: Complex tickets with related data grouped in objects

**Handling Arrays:**
- ✅ Storage devices are represented as an array of objects
- ✅ Each item in the array has the same structure
- ✅ Easy to iterate through and process programmatically

**Extracting Implied Information:**
- ✅ Urgency inferred from context ("blocking project deadline on Friday" → High)
- ✅ Suspected cause identified from symptoms described
- ✅ Device purpose categorized (OS drive vs. data drive)

💡 **Best Practice:** Use nested structures when data has clear relationships (device → components). This keeps data organized and easier to work with.

---

## Example 2: Multiple Error Extraction (JSON Array)

Sometimes a single message contains multiple distinct items that need to be extracted separately. In this example, a user reports five different errors throughout the day. We'll extract each error into its own object within a JSON array, maintaining consistent structure across all items while preserving the unique details of each error.

In [19]:
# Mock data: User reporting multiple different errors
multiple_errors_report = """
From: david.kumar@company.com
Subject: Multiple System Errors Today - Please Help

Hi IT Team,

I've been experiencing several different errors on my computer today (EMP-3392):

1. Around 9 AM, I got a Windows error saying "DRIVER_IRQL_NOT_LESS_OR_EQUAL"
   with a blue screen. The computer restarted automatically.

2. At 10:30 AM, Adobe Acrobat crashed with error code 0xc0000005 when I tried
   to open a PDF file from a client.

3. Just before lunch, my network printer showed "Error 49.FF04" and stopped
   printing completely. Other people can still print to it.

4. Around 2 PM, I got another blue screen with "SYSTEM_SERVICE_EXCEPTION" error.

5. Finally, at 3:15 PM, Outlook gave me error 0x80040600 and won't send emails now.

I'm really frustrated because this is affecting my work. These errors seem random
but I'm worried something serious is wrong with my computer.

Please help!
David Kumar
Sales Department
"""

Now let's create the extraction prompt and send it to the model:

In [20]:
# Extraction prompt for array structure
extraction_prompt = f"""
Extract ALL errors from this user report and format as JSON.

Return JSON with this structure:
{{
  "user_name": "string",
  "employee_id": "string",
  "department": "string",
  "report_date": "infer from context or use null",
  "total_errors": number,
  "user_sentiment": "string (frustrated/concerned/neutral/etc)",
  "errors": [
    {{
      "error_number": number,
      "error_code": "string or null",
      "error_message": "string",
      "source": "string (Windows/Application name/Hardware)",
      "timestamp_context": "string (time mentioned in report)",
      "error_type": "string (Blue Screen/Application Crash/Hardware Error/etc)"
    }}
  ]
}}

Extract each error into a separate array item with consistent structure.

User report:
{multiple_errors_report}

Return ONLY valid JSON.
"""

print("🔄 Extracting multiple errors into JSON array...")

# Make API call
response = client.responses.create(
    model=OPENAI_MODEL,
    input=extraction_prompt,
    text={"verbosity": "medium"}
)

errors_json = response.output_text.strip()
errors_data = json.loads(errors_json)

print(" Extracted Multiple Errors:")
print("=" * 80)
print(json.dumps(errors_data, indent=2))
print("=" * 80)

# Save to file
multiple_errors_path = "./multiple_errors.json"
with open(multiple_errors_path, 'w') as f:
    json.dump(errors_data, f, indent=2)

print(f" Saved to: {multiple_errors_path}")
print(f" Tokens used: {response.usage.total_tokens}")

# Display insights
print(f" 🔍 Analysis:")
print(f"  • User: {errors_data['user_name']} ({errors_data['department']})")
print(f"  • Total errors reported: {errors_data['total_errors']}")
print(f"  • User sentiment: {errors_data['user_sentiment']}")
print(f"  Error breakdown:")
for error in errors_data['errors']:
    print(f"    {error['error_number']}. {error['error_type']} - {error['source']}")
    print(f"       Time: {error['timestamp_context']}")
    if error['error_code']:
        print(f"       Code: {error['error_code']}")

🔄 Extracting multiple errors into JSON array...


 Extracted Multiple Errors:
{
  "user_name": "David Kumar",
  "employee_id": "EMP-3392",
  "department": "Sales Department",
  "report_date": null,
  "total_errors": 5,
  "user_sentiment": "frustrated",
  "errors": [
    {
      "error_number": 1,
      "error_code": "DRIVER_IRQL_NOT_LESS_OR_EQUAL",
      "error_message": "DRIVER_IRQL_NOT_LESS_OR_EQUAL",
      "source": "Windows",
      "timestamp_context": "Around 9 AM",
      "error_type": "Blue Screen"
    },
    {
      "error_number": 2,
      "error_code": "0xc0000005",
      "error_message": "0xc0000005",
      "source": "Adobe Acrobat",
      "timestamp_context": "10:30 AM",
      "error_type": "Application Crash"
    },
    {
      "error_number": 3,
      "error_code": "49.FF04",
      "error_message": "Error 49.FF04",
      "source": "Network Printer",
      "timestamp_context": "Just before lunch",
      "error_type": "Printer Error"
    },
    {
      "error_number": 4,
      "error_code": "SYSTEM_SERVICE_EXCEPTION",
     

### Key Points

**What we learned:**
- ✅ How to extract MULTIPLE similar items into a structured array
- ✅ Each array item has consistent structure (same fields)
- ✅ LLM can identify and separate distinct errors from narrative text
- ✅ We can include metadata (total count, sentiment) alongside the array
- ✅ Easy to process programmatically (loop through errors)

💡 **Use case:** Any scenario with multiple similar items - error logs, equipment lists, action items from meetings, multiple user requests in one message.

---

## Example 3: Hardware Inventory (CSV Format)

Now let's apply CSV extraction to a more complex scenario. We'll load a detailed inventory report from an external file - this simulates a realistic workflow where data comes from a report or document rather than being typed directly into code.

In [21]:
# Load detailed conference room inventory from file
with open("conference_rooms_detailed.txt", "r") as f:
    conference_rooms_description = f.read()

print("Loaded conference room inventory:")
print(conference_rooms_description[:200] + "...\n")

# Extraction prompt for CSV
extraction_prompt = f"""
Extract conference room equipment inventory as CSV.

Use these exact column headers:
room,device_type,manufacturer,model,serial_number,location_detail,quantity

Rules:
- One row per device item
- Include header row
- device_type should be category like "Display", "Camera", "Computer", "Phone System", etc.
- location_detail should include floor or mounting info if mentioned
- quantity should be 1 for individual items
- Use commas as separators
- Don't use quotes unless value contains comma

Text:
{conference_rooms_description}

Return ONLY the CSV data, no additional text.
"""

print("🔄 Extracting conference room inventory to CSV...")

# Make API call
response = client.responses.create(
    model=OPENAI_MODEL,
    input=extraction_prompt,
    text={"verbosity": "low"}
)

csv_output = response.output_text.strip()

print("Extracted CSV Data:")
print("=" * 100)
print(csv_output)
print("=" * 100)

# Save to file
inventory_csv_path = "./conference_room_inventory.csv"
with open(inventory_csv_path, 'w') as f:
    f.write(csv_output)

print(f"Saved to: {inventory_csv_path}")
print(f"Tokens used: {response.usage.total_tokens}")

Loaded conference room inventory:
Conference Room Inventory Report:

Executive Boardroom (3rd Floor):
- Two 75-inch Samsung QN75 displays mounted on the wall (serials: SAMQN-8821, SAMQN-8822)
- One Logitech Rally Bar camera system, se...

🔄 Extracting conference room inventory to CSV...


Extracted CSV Data:
room,device_type,manufacturer,model,serial_number,location_detail,quantity
Executive Boardroom,Display,Samsung,QN75,SAMQN-8821,"3rd Floor, mounted on wall",1
Executive Boardroom,Display,Samsung,QN75,SAMQN-8822,"3rd Floor, mounted on wall",1
Executive Boardroom,Camera,Logitech,Rally Bar,LOGI-RB-3345,3rd Floor,1
Executive Boardroom,Phone System,Polycom,RealPresence Trio 8800,POLY-8800-991,3rd Floor,1
Executive Boardroom,Computer,Dell,OptiPlex 7090,DELL-OPT-4455,3rd Floor,1
Room 2A,Display,LG,65-inch LG OLED,LG-OLED-2293,2nd Floor,1
Room 2A,Camera,Jabra,PanaCast,JAB-PC-7721,2nd Floor,1
Room 2A,Display,Microsoft,Surface Hub 2S 50-inch,MSFT-HUB-1203,2nd Floor,1
Room 2A,Accessory,,Wireless Presentation Adapter,WRLSS-001,2nd Floor,1
Room 2A,Accessory,,Wireless Presentation Adapter,WRLSS-002,2nd Floor,1
Training Room B,Display,Sony,Bravia 55-inch,SONY-BR-5501,1st Floor,1
Training Room B,Display,Sony,Bravia 55-inch,SONY-BR-5502,1st Floor,1
Training Room B,Display,Sony,Bravia

In [22]:
# Verify and analyze the CSV
print("🔍 Verifying CSV and analyzing inventory...")

# Read with pandas
df = pd.read_csv(inventory_csv_path)

print("✅ CSV is valid!")
print(f" Inventory Summary:")
print(f"  • Total items: {len(df)}")
print(f"  • Rooms covered: {df['room'].nunique()}")
print(f"  • Device types: {df['device_type'].nunique()}")
print(f" Items per room:")
print(df.groupby('room').size().to_string())
print(f" Device type breakdown:")
print(df.groupby('device_type').size().to_string())

print(f" Full Inventory:")

# Display as DataFrame (better formatting in Jupyter)
display(df)

print(f"💡 This CSV can be imported into Excel, Google Sheets, or an inventory database!")

🔍 Verifying CSV and analyzing inventory...
✅ CSV is valid!
 Inventory Summary:
  • Total items: 22
  • Rooms covered: 4
  • Device types: 7
 Items per room:
room
Executive Boardroom       5
Room 2A                   5
Small Meeting Room 105    3
Training Room B           9
 Device type breakdown:
device_type
Accessory             2
Camera                3
Computer              4
Conference System     1
Display              10
Docking Station       1
Phone System          1
 Full Inventory:


,room,device_type,manufacturer,model,serial_number,location_detail,quantity
0,Executive Boardroom,Display,Samsung,QN75,SAMQN-8821,"3rd Floor, mounted on wall",1
1,Executive Boardroom,Display,Samsung,QN75,SAMQN-8822,"3rd Floor, mounted on wall",1
2,Executive Boardroom,Camera,Logitech,Rally Bar,LOGI-RB-3345,3rd Floor,1
3,Executive Boardroom,Phone System,Polycom,RealPresence Trio 8800,POLY-8800-991,3rd Floor,1
4,Executive Boardroom,Computer,Dell,OptiPlex 7090,DELL-OPT-4455,3rd Floor,1
5,Room 2A,Display,LG,65-inch LG OLED,LG-OLED-2293,2nd Floor,1
6,Room 2A,Camera,Jabra,PanaCast,JAB-PC-7721,2nd Floor,1
7,Room 2A,Display,Microsoft,Surface Hub 2S 50-inch,MSFT-HUB-1203,2nd Floor,1
8,Room 2A,Accessory,NaN,Wireless Presentation Adapter,WRLSS-001,2nd Floor,1
9,Room 2A,Accessory,NaN,Wireless Presentation Adapter,WRLSS-002,2nd Floor,1


💡 This CSV can be imported into Excel, Google Sheets, or an inventory database!


---

# Data Validation

Extracted data needs validation before use in production systems. While LLMs are very good at extraction, they are not 100% accurate - the input data may be incomplete or ambiguous, and downstream systems expect specific formats.

Validation catches these issues early, before bad data reaches critical systems. It also provides feedback about what's missing or incorrect, helps track extraction quality over time, and flags uncertain extractions for human review.

Let's explore three key validation techniques:

## Technique 1: Required Fields Check

Ensure all critical fields are present and not empty.

In [23]:
def check_required_fields(data, required_fields):
    """
    Check if all required fields are present and not empty.

    Args:
        data (dict): Extracted data dictionary
        required_fields (list): List of field names that must be present

    Returns:
        dict: Validation result with is_valid flag and missing_fields list
    """
    missing_fields = []

    for field in required_fields:
        # Check if field exists
        if field not in data:
            missing_fields.append(field)
        # Check if field is empty (None, empty string, empty list)
        elif data[field] is None or data[field] == "" or data[field] == []:
            missing_fields.append(field)

    return {
        "is_valid": len(missing_fields) == 0,
        "missing_fields": missing_fields,
        "message": "All required fields present" if len(missing_fields) == 0 else f"Missing fields: {', '.join(missing_fields)}"
    }

# Example: Incomplete ticket data
incomplete_ticket = {
    "user_name": "Alex Johnson",
    "employee_id": "",  # Empty!
    "contact_email": "alex.johnson@company.com",
    "issue_summary": None,  # Missing!
    # department field completely missing
}

print("Testing Required Fields Validation")
print("Ticket data:")
print(json.dumps(incomplete_ticket, indent=2))

# Define what fields are required
required = ["user_name", "employee_id", "contact_email", "issue_summary", "department"]

# Validate
result = check_required_fields(incomplete_ticket, required)

print(f" Validation Result:")
print(f"  Valid: {result['is_valid']}")
print(f"  Message: {result['message']}")

if not result['is_valid']:
    print(f" ❌ Cannot process this ticket - missing critical information!")
    print(f"  Action needed: Request missing fields from user")

Testing Required Fields Validation
Ticket data:
{
  "user_name": "Alex Johnson",
  "employee_id": "",
  "contact_email": "alex.johnson@company.com",
  "issue_summary": null
}
 Validation Result:
  Valid: False
  Message: Missing fields: employee_id, issue_summary, department
 ❌ Cannot process this ticket - missing critical information!
  Action needed: Request missing fields from user


## Technique 2: Data Type & Format Validation

Check if data matches expected formats (email, employee ID, asset tag, etc.).

In [24]:
import re

def validate_data_formats(data):
    """
    Validate data formats for common IT fields.

    Args:
        data (dict): Extracted ticket data

    Returns:
        dict: Validation results with specific format errors
    """
    errors = []

    # Email format validation
    if 'contact_email' in data and data['contact_email']:
        email = data['contact_email']
        if '@' not in email or '.' not in email.split('@')[-1]:
            errors.append(f"Invalid email format: {email}")

    # Employee ID format validation (EMP-####)
    if 'employee_id' in data and data['employee_id']:
        emp_id = data['employee_id']
        if not re.match(r'^EMP-\\d{4}$', emp_id):
            errors.append(f"Invalid employee ID format: {emp_id} (expected: EMP-####)")

    # Asset tag validation (reasonable length)
    if 'asset_tag' in data and data['asset_tag']:
        asset_tag = data['asset_tag']
        if len(asset_tag) < 5 or len(asset_tag) > 20:
            errors.append(f"Asset tag length invalid: {asset_tag} (expected: 5-20 chars)")

    # Phone number validation (at least 7 digits)
    if 'contact_phone' in data and data['contact_phone']:
        phone = data['contact_phone']
        digits_only = re.sub(r'\\D', '', phone)  # Remove non-digits
        if len(digits_only) < 7:
            errors.append(f"Phone number too short: {phone}")

    return {
        "is_valid": len(errors) == 0,
        "errors": errors,
        "message": "All formats valid" if len(errors) == 0 else f"Found {len(errors)} format error(s)"
    }

# Example: Ticket with format errors
ticket_with_errors = {
    "user_name": "Patricia Lee",
    "employee_id": "12345",  # Wrong format (missing EMP- prefix)
    "contact_email": "patricia.lee.company.com",  # Missing @
    "contact_phone": "555",  # Too short
    "asset_tag": "PC",  # Too short
    "issue_summary": "Computer won't start"
}

print("Testing Format Validation")
print("Ticket data:")
print(json.dumps(ticket_with_errors, indent=2))

# Validate formats
result = validate_data_formats(ticket_with_errors)

print(f" Format Validation Result:")
print(f"  Valid: {result['is_valid']}")
print(f"  Message: {result['message']}")

if not result['is_valid']:
    print(f" ❌ Format Errors Detected:")
    for i, error in enumerate(result['errors'], 1):
        print(f"  {i}. {error}")
    print(f" ⚠️ Action needed: Data needs correction before processing")

Testing Format Validation
Ticket data:
{
  "user_name": "Patricia Lee",
  "employee_id": "12345",
  "contact_email": "patricia.lee.company.com",
  "contact_phone": "555",
  "asset_tag": "PC",
  "issue_summary": "Computer won't start"
}
 Format Validation Result:
  Valid: False
  Message: Found 4 format error(s)
 ❌ Format Errors Detected:
  1. Invalid email format: patricia.lee.company.com
  2. Invalid employee ID format: 12345 (expected: EMP-####)
  3. Asset tag length invalid: PC (expected: 5-20 chars)
  4. Phone number too short: 555
 ⚠️ Action needed: Data needs correction before processing


## Technique 3: Confidence Scoring

Ask the LLM to rate its own confidence and identify uncertain extractions.

In [25]:
# Mock data: Vague, incomplete ticket
vague_ticket = """
From: someone@company.com
Subject: Help

Something's wrong with my computer. It's not working right.
Can you fix it?
"""

Let's create the extraction prompt and send it to the model:

In [26]:
# Extraction with confidence scoring
extraction_prompt = f"""
Extract support ticket information from this email. Include a confidence score.

Return JSON with these fields:
- user_name: Extract if possible, use null if not found
- employee_id: Extract if mentioned, use null otherwise
- contact_email: Email address
- issue_summary: Brief summary of the issue
- urgency: Low/Medium/High/Critical (infer from context)
- confidence_score: Your confidence in this extraction (0-100)
- missing_fields: List of critical information that's missing
- notes: Any concerns or ambiguities about the extraction

Email:
{vague_ticket}

Return ONLY valid JSON.
"""

print("🔄 Extracting vague ticket with confidence scoring...")

response = client.responses.create(
    model=OPENAI_MODEL,
    input=extraction_prompt,
    text={"verbosity": "medium"}
)

extraction_json = response.output_text.strip()
extraction_data = json.loads(extraction_json)

print(" Extracted Data with Confidence:")
print("=" * 80)
print(json.dumps(extraction_data, indent=2))
print("=" * 80)

# Check confidence threshold
CONFIDENCE_THRESHOLD = 70

print(f" Confidence Analysis:")
print(f"  Confidence Score: {extraction_data.get('confidence_score', 0)}/100")
print(f"  Threshold: {CONFIDENCE_THRESHOLD}/100")

if extraction_data.get('confidence_score', 0) < CONFIDENCE_THRESHOLD:
    print(f"⚠️ LOW CONFIDENCE - Human Review Required!")
    print(f"  Missing fields: {', '.join(extraction_data.get('missing_fields', []))}")
    print(f"  Notes: {extraction_data.get('notes', 'N/A')}")
    print(f"  Action: Request more information from user")
else:
    print(f"✅ High confidence - Safe to process automatically")

print(f" Tokens used: {response.usage.total_tokens}")

🔄 Extracting vague ticket with confidence scoring...


 Extracted Data with Confidence:
{
  "user_name": "someone",
  "employee_id": null,
  "contact_email": "someone@company.com",
  "issue_summary": "Computer not functioning properly; user requests fix.",
  "urgency": "Medium",
  "confidence_score": 65,
  "missing_fields": [
    "employee_id",
    "urgency (unspecified in email)",
    "device_details (e.g., computer make/model/OS)",
    "preferred_contact_method"
  ],
  "notes": "No explicit name or urgency; user_name inferred from email local-part; minimal details about the issue and device."
}
 Confidence Analysis:
  Confidence Score: 65/100
  Threshold: 70/100
⚠️ LOW CONFIDENCE - Human Review Required!
  Missing fields: employee_id, urgency (unspecified in email), device_details (e.g., computer make/model/OS), preferred_contact_method
  Notes: No explicit name or urgency; user_name inferred from email local-part; minimal details about the issue and device.
  Action: Request more information from user
 Tokens used: 1356


---

# File Generation

Once you've extracted and validated data, you need to save it for downstream use. Saved files serve many purposes - importing into databases or ticketing systems, maintaining audit trails, batch processing multiple extractions, sharing structured data with colleagues, and aggregating data for reporting.

Let's explore three common file-saving patterns.

## Pattern 1: Saving JSON Files with Timestamps

Create unique filenames with timestamps to avoid overwriting data.

In [27]:
def save_json_with_timestamp(data, prefix="extracted_data"):
    """
    Save JSON data with timestamp in filename.

    Args:
        data (dict): Data to save
        prefix (str): Filename prefix

    Returns:
        str: Path to saved file
    """
    # Generate timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Create filename
    filename = f"./{prefix}_{timestamp}.json"

    # Save file
    with open(filename, 'w') as f:
        json.dump(data, f, indent=2)

    return filename

# Example: Save a ticket extraction
example_ticket = {
    "ticket_id": "TKT-2024-0157",
    "user_name": "Emily Chen",
    "employee_id": "EMP-8821",
    "department": "Finance",
    "issue_summary": "Excel crashes when opening large files",
    "urgency": "Medium",
    "extracted_at": datetime.now().isoformat()
}

print(" Saving JSON with timestamp...")

# Save the file
file_path = save_json_with_timestamp(example_ticket, prefix="ticket_extraction")

print(f"✅ Saved to: {file_path}")
print(f" File contents:")
with open(file_path, 'r') as f:
    print(f.read())

# Demonstrate saving multiple extractions
print("\\" + "="*80)
print("Saving multiple extractions...")

import time

for i in range(3):
    ticket = {
        "ticket_id": f"TKT-2024-{100 + i}",
        "user_name": f"User {i+1}",
        "issue": f"Issue {i+1}"
    }
    path = save_json_with_timestamp(ticket, prefix="ticket")
    print(f"  Saved: {path}")
    time.sleep(1)  # Small delay to ensure different timestamps

print("✅ Each file has unique timestamp - no overwrites!")

 Saving JSON with timestamp...
✅ Saved to: ./ticket_extraction_20260706_091929.json
 File contents:
{
  "ticket_id": "TKT-2024-0157",
  "user_name": "Emily Chen",
  "employee_id": "EMP-8821",
  "department": "Finance",
  "issue_summary": "Excel crashes when opening large files",
  "urgency": "Medium",
  "extracted_at": "2026-07-06T09:19:29.719861"
}
\================================================================================
Saving multiple extractions...
  Saved: ./ticket_20260706_091929.json


  Saved: ./ticket_20260706_091930.json


  Saved: ./ticket_20260706_091931.json


✅ Each file has unique timestamp - no overwrites!


## Pattern 2: Saving CSV Files

Convert extracted data to CSV format for spreadsheet compatibility.

In [28]:
def save_to_csv(data_list, filename, fieldnames=None):
    """
    Save list of dictionaries as CSV file.

    Args:
        data_list (list): List of dictionaries to save
        filename (str): Output filename
        fieldnames (list): Optional list of field names (uses first dict keys if None)

    Returns:
        str: Path to saved file
    """
    if not data_list:
        raise ValueError("data_list cannot be empty")

    # Use provided fieldnames or extract from first dictionary
    if fieldnames is None:
        fieldnames = list(data_list[0].keys())

    # Ensure filename includes path
    if not filename.startswith('./'):
        filename = f"./{filename}"

    # Write CSV
    with open(filename, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data_list)

    return filename

# Example: Save inventory data
inventory_items = [
    {
        "room": "Conference A",
        "device_type": "Display",
        "manufacturer": "Samsung",
        "model": "QN75",
        "serial_number": "SAM-001",
        "quantity": 1
    },
    {
        "room": "Conference A",
        "device_type": "Camera",
        "manufacturer": "Logitech",
        "model": "Rally Bar",
        "serial_number": "LOG-445",
        "quantity": 1
    },
    {
        "room": "Conference B",
        "device_type": "Display",
        "manufacturer": "LG",
        "model": "OLED65",
        "serial_number": "LG-2293",
        "quantity": 1
    }
]

print(" Saving inventory data to CSV...")

csv_path = save_to_csv(
    inventory_items,
    "inventory_export.csv",
    fieldnames=["room", "device_type", "manufacturer", "model", "serial_number", "quantity"]
)

print(f"✅ Saved to: {csv_path}")

# Read and display
df = pd.read_csv(csv_path)
print(f" CSV Contents ({len(df)} rows):")

# Display as DataFrame (better formatting in Jupyter)
display(df)

print(f"💡 This CSV can be opened in Excel or imported into inventory systems!")

 Saving inventory data to CSV...
✅ Saved to: ./inventory_export.csv
 CSV Contents (3 rows):


,room,device_type,manufacturer,model,serial_number,quantity
0,Conference A,Display,Samsung,QN75,SAM-001,1
1,Conference A,Camera,Logitech,Rally Bar,LOG-445,1
2,Conference B,Display,LG,OLED65,LG-2293,1


💡 This CSV can be opened in Excel or imported into inventory systems!


## Pattern 3: Appending to Existing Files

Build up a log file with multiple extractions over time.

In [29]:
def append_to_ticket_log(ticket_data, log_file="./ticket_log.json"):
    """
    Append ticket data to existing log file.

    Args:
        ticket_data (dict): Ticket data to append
        log_file (str): Path to log file

    Returns:
        dict: Status with total count
    """
    # Add timestamp to ticket
    ticket_data['logged_at'] = datetime.now().isoformat()

    # Check if file exists
    if os.path.exists(log_file):
        # Load existing data
        with open(log_file, 'r') as f:
            log_data = json.load(f)
    else:
        # Create new log structure
        log_data = {
            "log_created": datetime.now().isoformat(),
            "tickets": []
        }

    # Append new ticket
    log_data["tickets"].append(ticket_data)
    log_data["total_tickets"] = len(log_data["tickets"])
    log_data["last_updated"] = datetime.now().isoformat()

    # Save back to file
    with open(log_file, 'w') as f:
        json.dump(log_data, f, indent=2)

    return {
        "success": True,
        "total_tickets": log_data["total_tickets"],
        "log_file": log_file
    }

print(" Demonstrating ticket log appending...")

# Simulate processing multiple tickets
tickets_to_process = [
    {
        "ticket_id": "TKT-001",
        "user_name": "Alice Smith",
        "issue": "Password reset needed",
        "urgency": "Low"
    },
    {
        "ticket_id": "TKT-002",
        "user_name": "Bob Johnson",
        "issue": "VPN not connecting",
        "urgency": "High"
    },
    {
        "ticket_id": "TKT-003",
        "user_name": "Carol Davis",
        "issue": "Printer offline",
        "urgency": "Medium"
    }
]

# Process and log each ticket
for ticket in tickets_to_process:
    result = append_to_ticket_log(ticket)
    print(f"✅ Logged {ticket['ticket_id']} - Total tickets in log: {result['total_tickets']}")
    time.sleep(0.5)  # Small delay

# Read and display final log
print(f" Final Ticket Log:")
print("=" * 80)
with open("./ticket_log.json", 'r') as f:
    log_contents = json.load(f)

print(f"Log created: {log_contents['log_created']}")
print(f"Last updated: {log_contents['last_updated']}")
print(f"Total tickets: {log_contents['total_tickets']}\\n")

print("Tickets in log:")
for i, ticket in enumerate(log_contents['tickets'], 1):
    print(f"  {i}. {ticket['ticket_id']}: {ticket['issue']} (Urgency: {ticket['urgency']})")

print("=" * 80)
print(f" 💡 Log file grows with each append - perfect for ongoing ticket tracking!")

 Demonstrating ticket log appending...
✅ Logged TKT-001 - Total tickets in log: 1


✅ Logged TKT-002 - Total tickets in log: 2


✅ Logged TKT-003 - Total tickets in log: 3


 Final Ticket Log:
Log created: 2026-07-06T09:19:32.758841
Last updated: 2026-07-06T09:19:33.770920
Total tickets: 3\n
Tickets in log:
  1. TKT-001: Password reset needed (Urgency: Low)
  2. TKT-002: VPN not connecting (Urgency: High)
  3. TKT-003: Printer offline (Urgency: Medium)
 💡 Log file grows with each append - perfect for ongoing ticket tracking!


# Batch Processing

In real-world scenarios, you often need to process multiple items at once - clearing email backlogs, auditing existing tickets, or migrating data from old formats. Batch processing is more efficient than handling items one by one, and with progress tracking you can monitor how the job is progressing.



In [30]:
# Create array of 5 different mock support tickets
ticket_batch = [
    """From: anna.rodriguez@company.com
    Subject: Laptop Battery Draining Fast
    Hi, I'm Anna Rodriguez (EMP-3401) from Marketing. My laptop battery only lasts 30 minutes now.
    This is urgent as I have client meetings all day tomorrow. Please help!""",

    """From: kevin.park@company.com
    Subject: Cannot Connect to WiFi
    Kevin Park here, Sales dept, EMP-5623. My laptop won't connect to the office WiFi.
    I've tried restarting but it still doesn't work. Medium priority.""",

    """From: lisa.chen@company.com
    Subject: Printer Not Working
    Lisa Chen, Finance, EMP-7834. The printer on floor 3 shows 'Paper Jam' but there's no paper stuck.
    Multiple people are affected. Needs fixing ASAP!""",

    """From: marcus.johnson@company.com
    Subject: Slow Computer Performance
    Marcus Johnson, IT Support team member EMP-2019. My workstation is running extremely slow.
    All applications take forever to load. Low priority but annoying.""",

    """From: sophia.williams@company.com
    Subject: Email Account Locked
    Sophia Williams, HR department, EMP-9102. I got locked out after entering wrong password.
    Need access urgently for payroll processing today. CRITICAL!"""
]

In [31]:
# Batch processing

def batch_extract_tickets(ticket_list):
    """
    Process multiple tickets in batch with progress tracking.

    Args:
        ticket_list (list): List of ticket text strings

    Returns:
        dict: Results with successes, failures, and summary
    """
    results = []
    failures = []

    # Process with progress bar
    for i, ticket_text in enumerate(tqdm(ticket_list, desc="Processing tickets")):
        try:
            # Extraction prompt
            extraction_prompt = f"""
            Extract ticket information from this email. Return ONLY valid JSON.

            {{
              "ticket_id": "TKT-{datetime.now().strftime('%Y')}-{str(i+1).zfill(4)}",
              "user_name": "full name",
              "employee_id": "employee ID",
              "department": "department name",
              "contact_email": "email address",
              "issue_summary": "brief summary",
              "urgency": "Low/Medium/High/Critical"
            }}

            Email:
            {ticket_text}
            """

            # Make API call
            response = client.responses.create(
                model=OPENAI_MODEL,
                input=extraction_prompt,
                text={"verbosity": "low"}
            )

            # Parse JSON
            extracted_data = json.loads(response.output_text.strip())

            # Add metadata
            extracted_data['processed_at'] = datetime.now().isoformat()
            extracted_data['batch_index'] = i + 1

            results.append(extracted_data)

        except Exception as e:
            failures.append({
                "batch_index": i + 1,
                "error": str(e),
                "ticket_preview": ticket_text[:100] + "..."
            })

    return {
        "total_processed": len(ticket_list),
        "successful": len(results),
        "failed": len(failures),
        "success_rate": f"{(len(results)/len(ticket_list)*100):.1f}%",
        "results": results,
        "failures": failures
    }

print("🔄 Starting batch processing of 5 tickets...")

# Process the batch
batch_results = batch_extract_tickets(ticket_batch)

print(f"✅ Batch Processing Complete!")
print(f" Summary:")
print(f"  • Total tickets: {batch_results['total_processed']}")
print(f"  • Successful: {batch_results['successful']}")
print(f"  • Failed: {batch_results['failed']}")
print(f"  • Success rate: {batch_results['success_rate']}")

# Display successful extractions
if batch_results['results']:
    print(f"Successfully Extracted Tickets:")
    for ticket in batch_results['results']:
        print(f"  {ticket['ticket_id']}: {ticket['user_name']} - {ticket['issue_summary']} ({ticket['urgency']})")

# Display failures if any
if batch_results['failures']:
    print(f"❌ Failed Extractions:")
    for failure in batch_results['failures']:
        print(f"  Ticket {failure['batch_index']}: {failure['error']}")

🔄 Starting batch processing of 5 tickets...


Processing tickets:   0%|          | 0/5 [00:00<?, ?it/s]

Processing tickets:  20%|██        | 1/5 [00:06<00:24,  6.09s/it]

Processing tickets:  40%|████      | 2/5 [00:12<00:19,  6.45s/it]

Processing tickets:  60%|██████    | 3/5 [00:20<00:14,  7.11s/it]

Processing tickets:  80%|████████  | 4/5 [00:26<00:06,  6.53s/it]

Processing tickets: 100%|██████████| 5/5 [00:31<00:00,  6.09s/it]

Processing tickets: 100%|██████████| 5/5 [00:31<00:00,  6.33s/it]

✅ Batch Processing Complete!
 Summary:
  • Total tickets: 5
  • Successful: 5
  • Failed: 0
  • Success rate: 100.0%
Successfully Extracted Tickets:
  TKT-2026-0001: Anna Rodriguez - Laptop battery drains quickly; lasts ~30 minutes. (High)
  None: Kevin Park - Cannot connect to the office WiFi. (Medium)
  TKT-2026-0003: Lisa Chen - Printer on floor 3 shows 'Paper Jam' but no paper is stuck; issue affects multiple users; needs fixing ASAP. (High)
  TKT-2026-0004: Marcus Johnson - Workstation running extremely slow; all applications take forever to load. (Low)
  TKT-2026-0005: Sophia Williams - Account locked after incorrect password entry; urgent access needed for payroll processing today. (Critical)


In [32]:
# Save batch results to files
print("Saving batch results...")

# Save detailed JSON results
batch_json_path = "./batch_tickets.json"
with open(batch_json_path, 'w') as f:
    json.dump(batch_results, f, indent=2)
print(f"✅ Saved detailed results to: {batch_json_path}")

# Save as CSV for easy viewing
if batch_results['results']:
    # Prepare data for CSV (flatten structure)
    csv_data = []
    for ticket in batch_results['results']:
        csv_data.append({
            'ticket_id': ticket['ticket_id'],
            'user_name': ticket['user_name'],
            'employee_id': ticket['employee_id'],
            'department': ticket['department'],
            'contact_email': ticket['contact_email'],
            'issue_summary': ticket['issue_summary'],
            'urgency': ticket['urgency'],
            'processed_at': ticket['processed_at']
        })

    # Save to CSV
    batch_csv_path = "./batch_tickets.csv"
    df_batch = pd.DataFrame(csv_data)
    df_batch.to_csv(batch_csv_path, index=False)
    print(f"✅ Saved CSV summary to: {batch_csv_path}")

    # Display the DataFrame
    print(f" Batch Results (CSV view):")
    display(df_batch)
else:
    print("⚠️ No successful extractions to save to CSV")

print(f" Batch processing complete! Processed {batch_results['total_processed']} tickets.")

Saving batch results...
✅ Saved detailed results to: ./batch_tickets.json
✅ Saved CSV summary to: ./batch_tickets.csv
 Batch Results (CSV view):


,ticket_id,user_name,employee_id,department,contact_email,issue_summary,urgency,processed_at
0,TKT-2026-0001,Anna Rodriguez,EMP-3401,Marketing,anna.rodriguez@company.com,Laptop battery drains quickly; lasts ~30 minutes.,High,2026-07-06T09:19:40.409692
1,None,Kevin Park,EMP-5623,Sales,kevin.park@company.com,Cannot connect to the office WiFi.,Medium,2026-07-06T09:19:47.110698
2,TKT-2026-0003,Lisa Chen,EMP-7834,Finance,lisa.chen@company.com,Printer on floor 3 shows 'Paper Jam' but no pa...,High,2026-07-06T09:19:55.015541
3,TKT-2026-0004,Marcus Johnson,EMP-2019,IT Support,marcus.johnson@company.com,Workstation running extremely slow; all applic...,Low,2026-07-06T09:20:00.657760
4,TKT-2026-0005,Sophia Williams,EMP-9102,HR,sophia.williams@company.com,Account locked after incorrect password entry;...,Critical,2026-07-06T09:20:05.948186


 Batch processing complete! Processed 5 tickets.


---

# Error Handling & Edge Cases

Real-world data is messy. Users provide incomplete information, data contains contradictions, formats are ambiguous, and critical fields may be missing. A robust extraction system needs to detect these issues early, handle them gracefully with fallbacks, provide clear feedback, and flag uncertain cases for human review.

Let's demonstrate 5 common scenarios you'll encounter in production.

## Scenario 1: Missing Critical Information

When users send extremely vague requests like "my computer isn't working", most critical fields (name, employee ID, department) are completely missing. In this scenario, we ask the model to extract what it can, rate its own confidence, and suggest follow-up questions to ask the user for the missing information.

In [33]:
# Extremely vague ticket
vague_ticket = """
My computer isn't working. Please help!
"""

Now let's create the extraction prompt and send it to the model:

In [34]:
# Enhanced extraction prompt that handles missing info
extraction_prompt = f"""
Extract ticket information from this email. Handle missing information gracefully.

Return JSON:
{{
  "user_name": "name or null",
  "employee_id": "ID or null",
  "department": "department or null",
  "contact_email": "email or null",
  "issue_summary": "summary or 'Unspecified computer issue'",
  "urgency": "Low/Medium/High/Critical",
  "confidence_score": 0-100,
  "missing_fields": ["list", "of", "missing", "critical", "fields"],
  "follow_up_questions": ["questions to ask user for missing info"]
}}

Email:
{vague_ticket}

Return ONLY valid JSON.
"""

print("Scenario 1: Missing Critical Information")
print("Ticket text:")
print(vague_ticket)

response = client.responses.create(
    model=OPENAI_MODEL,
    input=extraction_prompt,
    text={"verbosity": "low"}
)

incomplete_data = json.loads(response.output_text.strip())

print(" Extracted Data:")
print(json.dumps(incomplete_data, indent=2))

# Save to file
incomplete_path = "./incomplete_ticket.json"
with open(incomplete_path, 'w') as f:
    json.dump(incomplete_data, f, indent=2)

# Analysis
print(f" Analysis:")
print(f"  Confidence Score: {incomplete_data.get('confidence_score', 0)}/100")
print(f"  Missing Fields: {', '.join(incomplete_data.get('missing_fields', []))}")

if incomplete_data.get('confidence_score', 0) < 70:
    print(f"⚠️ LOW CONFIDENCE - Action Required!")
    print(f"  Follow-up questions to ask user:")
    for i, question in enumerate(incomplete_data.get('follow_up_questions', []), 1):
        print(f"    {i}. {question}")
    print(f"  ❌ Cannot auto-process - needs human review")
else:
    print(f"✅ Sufficient information to proceed")

print(f" Saved to: {incomplete_path}")

Scenario 1: Missing Critical Information
Ticket text:

My computer isn't working. Please help!



 Extracted Data:
{
  "user_name": null,
  "employee_id": null,
  "department": null,
  "contact_email": null,
  "issue_summary": "My computer isn't working.",
  "urgency": "Medium",
  "confidence_score": 25,
  "missing_fields": [
    "user_name",
    "employee_id",
    "department",
    "contact_email",
    "urgency"
  ],
  "follow_up_questions": [
    "What is your full name?",
    "What is your employee ID?",
    "What department do you work in?",
    "What is your contact email or phone number?",
    "How urgent is this issue for you (Low/Medium/High/Critical)?",
    "Would you prefer on-site, remote, or phone support?"
  ]
}
 Analysis:
  Confidence Score: 25/100
  Missing Fields: user_name, employee_id, department, contact_email, urgency
⚠️ LOW CONFIDENCE - Action Required!
  Follow-up questions to ask user:
    1. What is your full name?
    2. What is your employee ID?
    3. What department do you work in?
    4. What is your contact email or phone number?
    5. How urgent is t

## Scenario 2: Contradictory Information

Sometimes users provide conflicting statements in the same message - for example, marking an issue as both "urgent" and "can wait until next week". Here we ask the model to detect these contradictions, list them explicitly, and explain how it resolved each one to arrive at a final answer.

In [35]:
# Ticket with contradictions
contradictory_ticket = """
From: james.anderson@company.com
Subject: Network Issue - Urgent but can wait

Hi IT,

I need help with a network problem. It's very urgent and critical, but it's also
low priority and can wait until next week. The issue is important but not really
a big deal.

James or maybe it's Jim Anderson (I go by both)
Employee ID: EMP-4501 or 4502, I forget which
"""

Now let's create the extraction prompt and send it to the model:

In [36]:
# Extraction prompt
extraction_prompt = f"""
Extract ticket information. Detect and handle contradictions.

Return JSON:
{{
  "user_name": "best judgment on name",
  "employee_id": "most likely ID",
  "issue_summary": "summary",
  "urgency": "best judgment: Low/Medium/High/Critical",
  "contradiction_detected": true or false,
  "contradictions_found": ["list of contradictions"],
  "resolution_notes": "how you resolved each contradiction"
}}

Email:
{contradictory_ticket}

Return ONLY valid JSON.
"""

print("Scenario 2: Contradictory Information")
print("Ticket text:")
print(contradictory_ticket)

response = client.responses.create(
    model=OPENAI_MODEL,
    input=extraction_prompt,
    text={"verbosity": "low"}
)

contradictory_data = json.loads(response.output_text.strip())

print(" Extracted Data:")
print(json.dumps(contradictory_data, indent=2))

# Save to file
contradictory_path = "./contradictory_ticket.json"
with open(contradictory_path, 'w') as f:
    json.dump(contradictory_data, f, indent=2)

# Analysis
print(f"Analysis:")
if contradictory_data.get('contradiction_detected'):
    print(f"  ⚠️ Contradictions Detected!")
    print(f"  Issues found:")
    for i, contradiction in enumerate(contradictory_data.get('contradictions_found', []), 1):
        print(f"    {i}. {contradiction}")
    print(f"  Resolution: {contradictory_data.get('resolution_notes', 'N/A')}")
    print(f"  ⚠️ Recommendation: Flag for human verification")
else:
    print(f"  ✅ No contradictions detected")

print(f" Saved to: {contradictory_path}")

Scenario 2: Contradictory Information
Ticket text:

From: james.anderson@company.com
Subject: Network Issue - Urgent but can wait

Hi IT,

I need help with a network problem. It's very urgent and critical, but it's also
low priority and can wait until next week. The issue is important but not really
a big deal.

James or maybe it's Jim Anderson (I go by both)
Employee ID: EMP-4501 or 4502, I forget which



 Extracted Data:
{
  "user_name": "James Anderson",
  "employee_id": "EMP-4501",
  "issue_summary": "Network issue; conflicting urgency statements: urgent/critical vs low priority and can wait until next week; no clear priority level.",
  "urgency": "High",
  "contradiction_detected": true,
  "contradictions_found": [
    "The email states the issue is 'urgent' and 'critical' (high urgency) but also says it is 'low priority' and can wait until next week.",
    "The subject mentions 'Urgent but can wait', which contradicts the urgency implied in the body."
  ],
  "resolution_notes": "Resolved by flagging contradictions between urgency indicators. Assigned an overall urgency of High based on explicit 'urgent' and 'critical' terms in body, while noting the conflict with statements that imply low priority and a delayed timeline."
}
Analysis:
  ⚠️ Contradictions Detected!
  Issues found:
    1. The email states the issue is 'urgent' and 'critical' (high urgency) but also says it is 'low pri

## Scenario 3: Ambiguous or Multiple Formats

Users often provide the same information in different formats - multiple email addresses, phone numbers with different formatting, or employee IDs with and without the prefix. In this scenario, we ask the model to standardize all values to a single consistent format while preserving the original variants for reference.  

In [37]:
# Ticket with ambiguous formats
ambiguous_ticket = """
From: rachel.kim@company.com OR rkim@company.com

My employee number is 5678 or EMP-5678, I'm not sure which format you need.
Phone: 555-0123 or (555) 012-3456 or +1-555-012-3456

Rachel
"""

We'll create the extraction prompt:

In [38]:
# Extraction prompt
extraction_prompt = f"""
Extract and STANDARDIZE data to consistent formats.

Standard formats to use:
- Employee ID: EMP-#### (4 digits with prefix)
- Phone: XXX-XXXX (simple format, remove country code and formatting)
- Email: Use primary corporate email

Return JSON:
{{
  "user_name": "name",
  "employee_id": "standardized to EMP-####",
  "contact_email": "primary email chosen",
  "contact_phone": "standardized format",
  "format_notes": "explain standardization decisions",
  "alternatives_found": {{
    "employee_id_variants": ["list", "of", "formats", "found"],
    "email_variants": ["list"],
    "phone_variants": ["list"]
  }}
}}

Email:
{ambiguous_ticket}

Return ONLY valid JSON.
"""

print("Scenario 3: Ambiguous or Multiple Formats")
print("Ticket text:")
print(ambiguous_ticket)

response = client.responses.create(
    model=OPENAI_MODEL,
    input=extraction_prompt,
    text={"verbosity": "low"}
)

ambiguous_data = json.loads(response.output_text.strip())

print("Extracted & Standardized Data:")
print(json.dumps(ambiguous_data, indent=2))

# Save to file
ambiguous_path = "./ambiguous_formats.json"
with open(ambiguous_path, 'w') as f:
    json.dump(ambiguous_data, f, indent=2)

# Analysis
print(f"Standardization Results:")
print(f"  Employee ID: {ambiguous_data.get('employee_id')}")
print(f"  Email: {ambiguous_data.get('contact_email')}")
print(f"  Phone: {ambiguous_data.get('contact_phone')}")
print(f"  Notes: {ambiguous_data.get('format_notes', 'N/A')}")

print(f"✅ Data standardized to consistent formats")
print(f" Saved to: {ambiguous_path}")

Scenario 3: Ambiguous or Multiple Formats
Ticket text:

From: rachel.kim@company.com OR rkim@company.com

My employee number is 5678 or EMP-5678, I'm not sure which format you need.
Phone: 555-0123 or (555) 012-3456 or +1-555-012-3456

Rachel



Extracted & Standardized Data:
{
  "user_name": "Rachel",
  "employee_id": "EMP-5678",
  "contact_email": "rachel.kim@company.com",
  "contact_phone": "555-0123",
  "format_notes": "Standardized employee_id to EMP-5678. Chose primary email as rachel.kim@company.com. Standardized phone to XXX-XXXX by removing country code (+1) and non-digit formatting; kept local 555-0123.",
  "alternatives_found": {
    "employee_id_variants": [
      "5678",
      "EMP-5678"
    ],
    "email_variants": [
      "rachel.kim@company.com",
      "rkim@company.com"
    ],
    "phone_variants": [
      "555-0123",
      "(555) 012-3456",
      "+1-555-012-3456"
    ]
  }
}
Standardization Results:
  Employee ID: EMP-5678
  Email: rachel.kim@company.com
  Phone: 555-0123
  Notes: Standardized employee_id to EMP-5678. Chose primary email as rachel.kim@company.com. Standardized phone to XXX-XXXX by removing country code (+1) and non-digit formatting; kept local 555-0123.
✅ Data standardized to consistent form

## Scenario 4: Invalid Data Detected

Not all extracted data is correct. Here we intentionally extract data "as-is" without letting the model fix it, and then run our own validation function to catch format errors (invalid emails, wrong ID patterns, short asset tags). This demonstrates why post-extraction validation is critical - the model extracts faithfully, but validation catches the problems.    

In [39]:
# Ticket with invalid data
invalid_ticket = """
From: bad.email.format
Employee: 99
Asset Tag: PC
Contact: jane.doe@company.com

My computer won't start. Please help!
"""

Now let's create the extraction prompt and send it to the model:

In [40]:
# Comprehensive validation function
def comprehensive_validation(data):
    """Validate extracted data against business rules."""
    validation_result = {
        "is_valid": True,
        "errors": [],
        "warnings": []
    }

    # Email validation
    if 'contact_email' in data and data['contact_email']:
        if '@' not in data['contact_email']:
            validation_result['errors'].append("Invalid email: missing @")
            validation_result['is_valid'] = False

    # Employee ID format
    if 'employee_id' in data and data['employee_id']:
        if not re.match(r'^EMP-\d{4}$', data['employee_id']):
            validation_result['errors'].append(f"Invalid employee ID format: {data['employee_id']} (expected: EMP-####)")
            validation_result['is_valid'] = False

    # Asset tag length
    if 'asset_tag' in data and data['asset_tag']:
        if len(data['asset_tag']) < 5:
            validation_result['warnings'].append(f"Asset tag too short: {data['asset_tag']} (expected: 5-20 chars)")

    return validation_result

# Extract with explicit instruction to extract AS-IS
print("Scenario 4: Invalid Data Detected")
print("Ticket text (with invalid data):")
print(invalid_ticket)

extraction_prompt = f"""
Extract ticket data EXACTLY as written, even if formats look wrong.
Do NOT skip or correct invalid data - extract it as-is so validation can catch issues.

Return JSON:
{{
  "contact_email": "extract email EXACTLY as written (even if invalid)",
  "employee_id": "extract ID EXACTLY as written (even if wrong format)",
  "asset_tag": "extract tag EXACTLY as written (even if too short)",
  "issue_summary": "summary"
}}

IMPORTANT: Extract data as-is, don't validate or correct it.

Ticket:
{invalid_ticket}
"""

response = client.responses.create(
    model=OPENAI_MODEL,
    input=extraction_prompt,
    text={"verbosity": "low"}
)

invalid_data = json.loads(response.output_text.strip())

print("Extracted Data (as-is, unvalidated):")
print(json.dumps(invalid_data, indent=2))

# Run validation
validation = comprehensive_validation(invalid_data)

print(f"🔍 Validation Results:")
print(f"  Valid: {validation['is_valid']}")

if validation['errors']:
    print(f" ❌ Errors ({len(validation['errors'])}):")
    for error in validation['errors']:
        print(f"    • {error}")

if validation['warnings']:
    print(f"  ⚠️ Warnings ({len(validation['warnings'])}):")
    for warning in validation['warnings']:
        print(f"    • {warning}")

# Save both data and validation report
combined_result = {
    "extracted_data": invalid_data,
    "validation": validation
}

invalid_path = "./invalid_data_validation.json"
with open(invalid_path, 'w') as f:
    json.dump(combined_result, f, indent=2)

print(f" Saved data and validation report to: {invalid_path}")

if not validation['is_valid']:
    print(f"⛔ Action Required: Data needs correction before processing")
    print(f"💡 This demonstrates why validation is critical:")
    print(f"   The LLM extracted data as-is, and validation caught the problems!")
else:
    print(f"✅ All validations passed")

Scenario 4: Invalid Data Detected
Ticket text (with invalid data):

From: bad.email.format
Employee: 99
Asset Tag: PC
Contact: jane.doe@company.com

My computer won't start. Please help!



Extracted Data (as-is, unvalidated):
{
  "contact_email": "jane.doe@company.com",
  "employee_id": "99",
  "asset_tag": "PC",
  "issue_summary": "My computer won't start. Please help!"
}
🔍 Validation Results:
  Valid: False
 ❌ Errors (1):
    • Invalid employee ID format: 99 (expected: EMP-####)
  ⚠️ Warnings (1):
    • Asset tag too short: PC (expected: 5-20 chars)
 Saved data and validation report to: ./invalid_data_validation.json
⛔ Action Required: Data needs correction before processing
💡 This demonstrates why validation is critical:
   The LLM extracted data as-is, and validation caught the problems!


## Scenario 5: Partial Success

Some reports are so vague that only fragments of information can be extracted. For example, a secondhand report like "something wrong in room B, Sarah mentioned it". Here we ask the model to be transparent about what it could and couldn't determine, rate its confidence, and recommend concrete next steps for follow-up.      

In [41]:
# Extremely vague secondhand report
partial_ticket = """
Something wrong in room B. Sarah mentioned it earlier.
"""

Now let's create the extraction prompt and send it to the model:

In [42]:
extraction_prompt = f"""
Extract what you can from this vague report. Be honest about what's unknown.

Return JSON:
{{
  "extraction_status": "complete/partial/minimal",
  "successfully_extracted": {{
    "location": "value or null",
    "reported_by": "value or null",
    "issue_summary": "value or null"
  }},
  "failed_to_extract": ["list", "of", "fields", "that", "couldn't", "be", "determined"],
  "confidence_score": 0-100,
  "assumptions_made": ["list any assumptions"],
  "recommended_actions": ["what to do next"]
}}

Report:
{partial_ticket}

Return ONLY valid JSON.
"""

print("Scenario 5: Partial Success")
print("Ticket text (very vague secondhand report):")
print(partial_ticket)

response = client.responses.create(
    model=OPENAI_MODEL,
    input=extraction_prompt,
    text={"verbosity": "low"}
)

partial_data = json.loads(response.output_text.strip())

print("Extraction Result:")
print(json.dumps(partial_data, indent=2))

# Save to file
partial_path = "./partial_extraction.json"
with open(partial_path, 'w') as f:
    json.dump(partial_data, f, indent=2)

# Analysis
print(f"Analysis:")
print(f"  Extraction Status: {partial_data.get('extraction_status', 'unknown')}")
print(f"  Confidence: {partial_data.get('confidence_score', 0)}/100")

print(f"  ✅ Successfully Extracted:")
for field, value in partial_data.get('successfully_extracted', {}).items():
    if value:
        print(f"    • {field}: {value}")

print(f"  ❌ Could Not Extract:")
for field in partial_data.get('failed_to_extract', []):
    print(f"    • {field}")

print(f"Recommended Actions:")
for i, action in enumerate(partial_data.get('recommended_actions', []), 1):
    print(f"    {i}. {action}")

print(f"⚠️ Status: Partial extraction only - human follow-up required")
print(f"Saved to: {partial_path}")

Scenario 5: Partial Success
Ticket text (very vague secondhand report):

Something wrong in room B. Sarah mentioned it earlier.



Extraction Result:
{
  "extraction_status": "partial",
  "successfully_extracted": {
    "location": "Room B",
    "reported_by": "Sarah",
    "issue_summary": "Something wrong in Room B"
  },
  "failed_to_extract": [],
  "confidence_score": 75,
  "assumptions_made": [
    "Interpreted 'room B' as the location",
    "Assumed Sarah is the person who reported the issue",
    "Kept the issue_summary as stated without inferring a specific problem"
  ],
  "recommended_actions": [
    "Visually inspect Room B to identify the issue",
    "Ask Sarah for any additional details or context",
    "Document findings and determine safety implications",
    "Assign maintenance or facilities staff to address"
  ]
}
Analysis:
  Extraction Status: partial
  Confidence: 75/100
  ✅ Successfully Extracted:
    • location: Room B
    • reported_by: Sarah
    • issue_summary: Something wrong in Room B
  ❌ Could Not Extract:
Recommended Actions:
    1. Visually inspect Room B to identify the issue
    2. Ask 

## Comprehensive Error Handler

Combine all techniques into a robust extraction function.

In [43]:
def safe_extract_with_validation(ticket_text):
    """
    Master extraction function with comprehensive error handling.

    Returns: (extracted_data, overall_status, report)
    - overall_status: "ready", "needs_review", or "requires_action"
    """
    try:
        # Enhanced extraction with confidence and validation
        extraction_prompt = f"""
        Extract ticket data with quality indicators.

        Return JSON:
        {{
          "user_name": "name or null",
          "employee_id": "ID or null",
          "department": "dept or null",
          "contact_email": "email or null",
          "issue_summary": "summary",
          "urgency": "Low/Medium/High/Critical",
          "confidence_score": 0-100,
          "missing_critical_fields": ["list"],
          "data_quality_notes": "any concerns or issues"
        }}

        Ticket:
        {ticket_text}
        """

        response = client.responses.create(
            model=OPENAI_MODEL,
            input=extraction_prompt,
            text={"verbosity": "low"}
        )

        extracted = json.loads(response.output_text.strip())

        # Validate formats
        validation = comprehensive_validation(extracted)

        # Check confidence
        confidence = extracted.get('confidence_score', 0)

        # Determine status
        if validation['is_valid'] and confidence >= 80 and not extracted.get('missing_critical_fields'):
            status = "ready"
        elif validation['is_valid'] and confidence >= 60:
            status = "needs_review"
        else:
            status = "requires_action"

        # Generate report
        report = {
            "extraction_status": status,
            "confidence_score": confidence,
            "validation_passed": validation['is_valid'],
            "validation_errors": validation.get('errors', []),
            "validation_warnings": validation.get('warnings', []),
            "missing_fields": extracted.get('missing_critical_fields', []),
            "notes": extracted.get('data_quality_notes', '')
        }

        return extracted, status, report

    except Exception as e:
        return None, "error", {"error": str(e), "message": "Extraction failed"}

# Test with different ticket qualities
test_tickets = [
    ("Complete ticket", """From: john.doe@company.com
    I'm John Doe (EMP-1234) from Sales. My laptop won't connect to WiFi. High priority please!"""),

    ("Vague ticket", """Computer broken. Help!"""),

    ("Medium quality", """Sarah from HR, laptop slow, not too urgent""")
]

print("Testing Comprehensive Error Handler")
print("=" * 80)

for label, ticket in test_tickets:
    print(f" Test: {label}")
    print(f"Ticket: {ticket[:60]}...")

    data, status, report = safe_extract_with_validation(ticket)

    print(f"  Status: {status.upper()}")
    print(f"  Confidence: {report.get('confidence_score', 'N/A')}/100")

    if status == "ready":
        print(f"  ✅ Ready for automatic processing")
    elif status == "needs_review":
        print(f"  ⚠️ Needs human review before processing")
    elif status == "requires_action":
        print(f"  ❌ Requires user follow-up")
        if report.get('missing_fields'):
            print(f"  Missing: {', '.join(report['missing_fields'])}")

    print("-" * 80)

Testing Comprehensive Error Handler
 Test: Complete ticket
Ticket: From: john.doe@company.com
    I'm John Doe (EMP-1234) from ...


  Status: READY
  Confidence: 92/100
  ✅ Ready for automatic processing
--------------------------------------------------------------------------------
 Test: Vague ticket
Ticket: Computer broken. Help!...


  Status: NEEDS_REVIEW
  Confidence: 60/100
  ⚠️ Needs human review before processing
--------------------------------------------------------------------------------
 Test: Medium quality
Ticket: Sarah from HR, laptop slow, not too urgent...


  Status: NEEDS_REVIEW
  Confidence: 72/100
  ⚠️ Needs human review before processing
--------------------------------------------------------------------------------


---

# Mini-Project: Support Ticket Intake System

Let's build a complete end-to-end system that processes support requests.

---

## System Requirements

Our system will:
1. Take raw support request text
2. Extract structured information
3. Validate the extraction
4. Check confidence and completeness
5. Save to appropriate files (JSON + CSV)
6. Generate formatted ticket summary
7. Provide feedback if information is insufficient


In [44]:
def process_support_request(request_text):
    """
    Complete ticket intake system - processes support request end-to-end.

    Args:
        request_text (str): Raw support request

    Returns:
        dict: Processing result with status and details
    """
    print(f"Processing Support Request...")
    print(f"{'='*80}")

    # Step 1: Extract structured information
    extraction_prompt = f"""
    Extract complete support ticket information.

    Return JSON:
    {{
      "ticket_id": "TKT-{datetime.now().strftime('%Y%m%d')}-{datetime.now().strftime('%H%M%S')}",
      "user_name": "name or null",
      "employee_id": "ID or null",
      "department": "dept or null",
      "contact_email": "email or null",
      "contact_phone": "phone or null",
      "issue_summary": "summary",
      "issue_details": "detailed description",
      "urgency": "Low/Medium/High/Critical",
      "confidence_score": 0-100,
      "missing_info": ["list of missing fields"]
    }}

    Request:
    {request_text}
    """

    try:
        # Extract
        response = client.responses.create(
            model=OPENAI_MODEL,
            input=extraction_prompt,
            text={"verbosity": "low"}
        )

        ticket_data = json.loads(response.output_text.strip())
        ticket_data['processed_at'] = datetime.now().isoformat()
        ticket_data['raw_request'] = request_text

        # Step 2: Validate
        validation = comprehensive_validation(ticket_data)

        # Step 3: Determine status
        confidence = ticket_data.get('confidence_score', 0)
        missing = ticket_data.get('missing_info', [])

        if validation['is_valid'] and confidence >= 80 and len(missing) == 0:
            status = "Ready"
            action = "Auto-process"
        elif validation['is_valid'] and confidence >= 60:
            status = "Needs Review"
            action = "Human verification recommended"
        else:
            status = "Requires Action"
            action = "Request additional information from user"

        ticket_data['processing_status'] = status
        ticket_data['recommended_action'] = action
        ticket_data['validation_result'] = validation

        # Step 4: Save to files
        # JSON (detailed)
        json_path = f"./processed_tickets_log.json"

        # Load existing or create new
        if os.path.exists(json_path):
            with open(json_path, 'r') as f:
                log = json.load(f)
        else:
            log = {"tickets": [], "total_count": 0}

        log['tickets'].append(ticket_data)
        log['total_count'] = len(log['tickets'])
        log['last_updated'] = datetime.now().isoformat()

        with open(json_path, 'w') as f:
            json.dump(log, f, indent=2)

        # CSV (summary)
        csv_path = "./processed_tickets_summary.csv"
        csv_data = {
            'ticket_id': ticket_data['ticket_id'],
            'user_name': ticket_data.get('user_name', 'Unknown'),
            'department': ticket_data.get('department', 'Unknown'),
            'issue_summary': ticket_data['issue_summary'],
            'urgency': ticket_data['urgency'],
            'status': status,
            'confidence': confidence,
            'processed_at': ticket_data['processed_at']
        }

        # Append to CSV
        file_exists = os.path.exists(csv_path)
        with open(csv_path, 'a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=csv_data.keys())
            if not file_exists:
                writer.writeheader()
            writer.writerow(csv_data)

        # Step 5: Generate summary
        print(f"✅ TICKET PROCESSED")
        print(f"Ticket ID: {ticket_data['ticket_id']}")
        print(f"Status: {status}")
        print(f"Confidence: {confidence}/100")
        print(f"Action: {action}")

        print(f" Ticket Details:")
        print(f"  User: {ticket_data.get('user_name', 'Unknown')}")
        print(f"  Department: {ticket_data.get('department', 'Unknown')}")
        print(f"  Issue: {ticket_data['issue_summary']}")
        print(f"  Urgency: {ticket_data['urgency']}")

        if missing:
            print(f"⚠️ Missing Information:")
            for item in missing:
                print(f"  • {item}")

        if validation['errors']:
            print(f"❌ Validation Errors:")
            for error in validation['errors']:
                print(f"  • {error}")

        print(f"Saved to:")
        print(f"  • {json_path} (detailed log)")
        print(f"  • {csv_path} (summary)")

        print(f"{'='*80}")

        return {
            "success": True,
            "ticket_id": ticket_data['ticket_id'],
            "status": status,
            "ticket_data": ticket_data
        }

    except Exception as e:
        print(f"❌ Processing failed: {e}")
        return {
            "success": False,
            "error": str(e)
        }

print("✅ Support Ticket Intake System Ready!")

✅ Support Ticket Intake System Ready!


In [45]:
# Test Case 1: Complete, clear ticket
test1 = """
From: alex.morgan@company.com
Employee ID: EMP-5521
Department: Engineering

Hi IT Support,

My Dell Latitude laptop is overheating and shutting down randomly during video calls.
This is high priority as I have important client demos this week.

You can reach me at ext. 3344 or this email.

Thanks,
Alex Morgan
"""

result1 = process_support_request(test1)

Processing Support Request...


✅ TICKET PROCESSED
Ticket ID: TKT-20260706-092115
Status: Ready
Confidence: 92/100
Action: Auto-process
 Ticket Details:
  User: Alex Morgan
  Department: Engineering
  Issue: Dell Latitude laptop overheating and shutting down randomly during video calls
  Urgency: High
Saved to:
  • ./processed_tickets_log.json (detailed log)
  • ./processed_tickets_summary.csv (summary)


In [46]:
# Test Case 2: Vague ticket
test2 = """
Computer not working properly. Need help.
"""

result2 = process_support_request(test2)

Processing Support Request...


✅ TICKET PROCESSED
Ticket ID: TKT-20260706-092124
Status: Requires Action
Confidence: 25/100
Action: Request additional information from user
 Ticket Details:
  User: None
  Department: None
  Issue: Computer not working properly
  Urgency: Medium
⚠️ Missing Information:
  • user_name
  • employee_id
  • department
  • contact_email
  • contact_phone
  • issue_details
Saved to:
  • ./processed_tickets_log.json (detailed log)
  • ./processed_tickets_summary.csv (summary)


In [47]:
# Test Case 3: Complex ticket with nested device info
test3 = """
From: tech.lead@company.com
Subject: Workstation GPU Issues

Hey IT,

I'm the tech lead for the design team (EMP-8934). Our main rendering workstation
(Dell Precision 7920 with NVIDIA Quadro RTX 8000, 256GB RAM) keeps crashing during
3D rendering jobs. Critical priority - we have a project deadline tomorrow!

The system has dual Xeon processors and runs Windows 11 Pro.

Contact: maria.santos@company.com or 555-0199

Maria Santos
"""

result3 = process_support_request(test3)

Processing Support Request...


✅ TICKET PROCESSED
Ticket ID: TKT-20260706-092144
Status: Ready
Confidence: 92/100
Action: Auto-process
 Ticket Details:
  User: Maria Santos
  Department: Design
  Issue: Workstation GPU issues - crashes during 3D rendering
  Urgency: Critical
Saved to:
  • ./processed_tickets_log.json (detailed log)
  • ./processed_tickets_summary.csv (summary)


In [48]:
# View the processed tickets summary
print("PROCESSED TICKETS SUMMARY")
print("=" * 80)

# Load and display CSV summary
if os.path.exists("./processed_tickets_summary.csv"):
    df_summary = pd.read_csv("./processed_tickets_summary.csv")
    print(f"Total tickets processed: {len(df_summary)}")
    display(df_summary)
else:
    print("No tickets processed yet.")

PROCESSED TICKETS SUMMARY
Total tickets processed: 3


,ticket_id,user_name,department,issue_summary,urgency,status,confidence,processed_at
0,TKT-20260706-092115,Alex Morgan,Engineering,Dell Latitude laptop overheating and shutting ...,High,Ready,92,2026-07-06T09:21:23.999037
1,TKT-20260706-092124,NaN,NaN,Computer not working properly,Medium,Requires Action,25,2026-07-06T09:21:44.400534
2,TKT-20260706-092144,Maria Santos,Design,Workstation GPU issues - crashes during 3D ren...,Critical,Ready,92,2026-07-06T09:21:54.129693


---

# Best Practices & Key Takeaways



### 1. Always Validate Extracted Data
- ✅ Check required fields are present and not empty
- ✅ Validate formats (email, IDs, phone numbers)
- ✅ Use confidence scores to flag uncertain extractions
- ✅ Implement validation before using data downstream

### 2. Choose the Right Model
- ✅ **gpt-5-nano** works well for most extraction tasks (cost-efficient)
- ✅ Only upgrade to premium models when quality difference matters
- ✅ Test with cheaper models first

### 3. Use Appropriate Output Formats
- ✅ **CSV**: Asset lists, inventories, simple databases
- ✅ **JSON**: Support tickets, complex records, API integration
- ✅ **Pydantic**: Production systems, database integration, strict validation

### 4. Handle Missing Information Gracefully
- ✅ Use `null` for missing fields instead of guessing
- ✅ Ask LLM to list missing_fields in response
- ✅ Generate follow-up questions for users
- ✅ Set confidence thresholds (e.g., < 70 = needs review)

### 5. Save Extractions for Record-Keeping
- ✅ Use timestamps in filenames to prevent overwrites
- ✅ Save detailed data (JSON) AND summary data (CSV)
- ✅ Append to log files for ongoing tracking
- ✅ Include metadata (processed_at, confidence, etc.)

### 6. Batch Process When Possible
- ✅ More efficient than one-by-one
- ✅ Use progress indicators (tqdm)
- ✅ Track successes and failures separately
- ✅ Generate summary reports

### 7. Implement Error Handling
- ✅ Always use try/except blocks
- ✅ Handle rate limits gracefully
- ✅ Provide helpful error messages
- ✅ Log errors for debugging

### 8. Structure Prompts Effectively
- ✅ Be specific about desired output format
- ✅ Provide example structure with concrete values
- ✅ Use "Return ONLY valid JSON" for structured output
- ✅ Set verbosity to "low" for cleaner JSON responses

---

## When to Use Each Format

| Format | Best For | Pros | Cons |
|--------|----------|------|------|
| **CSV** | Equipment lists, simple inventories, Excel reports | Easy to open in Excel, simple structure | No nesting, all strings |
| **JSON** | Support tickets, complex data, API integration | Flexible, supports nesting, native types | More verbose |
| **Pydantic** | Production systems, strict validation | Type safety, auto-validation, clear errors | Requires schema definition |

---

## Real-World Applications

**1. Automated Ticket Field Population**
- Extract user info, issue details, urgency from emails
- Populate ticketing system fields automatically
- Save support agents 2-3 minutes per ticket

**2. Knowledge Base Creation**
- Extract structured data from support emails
- Build searchable knowledge base
- Identify common issues and solutions

**3. Asset Tracking & Inventory**
- Extract equipment details from reports
- Maintain up-to-date inventory databases
- Track hardware across locations

**4. Meeting Notes to Action Items**
- Extract tasks, owners, deadlines from meeting notes
- Convert to structured task lists
- Integrate with project management tools

**5. Error Log Analysis**
- Extract error codes, sources, timestamps from logs
- Categorize and prioritize issues
- Generate incident reports



---

# Exercises

Practice what you've learned with these hands-on exercises!





### Exercise 1: Network Incident Report Extraction

**Scenario:** Your network team sends incident reports via email. Extract structured data from these reports.

**Mock Incident Report:**
```
Network Incident Report
Date: January 15, 2025, 14:30
Location: Building C, Floor 3
Affected Systems: File Server FS-301, Database Server DB-205
Issue: Complete network outage affecting 45 users
Duration: 2 hours 15 minutes
Cause: Failed network switch (SW-C3-12)
Resolution: Replaced faulty switch, restored connectivity
Reported by: Network Admin - Tom Chen
```

**Your Task:**
1. Create an extraction prompt for incident data
2. Extract to JSON with these fields:
   - `incident_date`
   - `incident_time`
   - `location`
   - `affected_systems` (array)
   - `user_impact` (number of users)
   - `duration`
   - `root_cause`
   - `resolution`
   - `reported_by`
3. Save to `/content/network_incident.json`
4. Display the results

**Bonus:** Add severity classification (Low/Medium/High/Critical) based on user impact

---

### Exercise 2: Equipment Checkout System

**Scenario:** Build a system to track equipment checkouts from inventory.

**Mock Checkout Requests:**
```
Request 1:
"Hi, I'm Jessica Lee from Marketing (EMP-4401). I need to borrow a MacBook Pro
and a wireless presenter for a client presentation on Jan 20th. I'll return
them by Jan 22nd. My email is jlee@company.com"

Request 2:
"Tom Wilson here, IT dept, employee 5623. Need the Nikon camera and tripod
for documenting the new server room setup. Checkout: tomorrow, return: 3 days later"

Request 3:
"Can I get a laptop? Sarah from Sales. Need it for the trade show next week."
```

**Your Task:**
1. Extract checkout information from each request
2. Create JSON with fields:
   - `requester_name`
   - `employee_id`
   - `department`
   - `contact_email`
   - `items_requested` (array of equipment)
   - `checkout_date`
   - `return_date`
   - `purpose`
3. Save each to CSV: `/content/equipment_checkout.csv`
4. Include columns: requester, items, checkout_date, return_date, status
5. Mark incomplete requests with status "pending_info"

**Bonus:** Calculate number of days for each checkout and flag if > 7 days

---

### Exercise 3: Extend the Mini-Project

**Scenario:** Enhance the Support Ticket Intake System with additional features.

**Your Tasks:**

### Part A: Email Notification Generation
After extracting ticket data, generate a formatted email notification.

**Requirements:**
1. Create a function `generate_ticket_email(ticket_data)`
2. Email should include:
   - Subject line with ticket ID and urgency
   - Greeting with user name
   - Ticket summary
   - Expected response time based on urgency
   - Contact information for follow-up
3. Save email text to `/content/ticket_email_{ticket_id}.txt`

**Example Output:**
```
Subject: [TKT-20250115-001] High Priority Ticket Created

Hello Alex Morgan,

Your support ticket has been created:

Ticket ID: TKT-20250115-001
Issue: Laptop overheating during video calls
Urgency: High
Expected Response: Within 4 hours

Our team will contact you at alex.morgan@company.com or ext. 3344

Thank you,
IT Support Team
```

### Part B: Severity Scoring
Add automatic severity scoring based on extracted information.

**Requirements:**
1. Create scoring function that assigns points:
   - Urgency: Critical=10, High=7, Medium=4, Low=2
   - Missing info: -2 points per missing field
   - Keywords in issue: "crash"=+3, "data loss"=+5, "slow"=+1
2. Calculate total severity score (0-20)
3. Add to ticket data and save

### Part C: Automatic Team Assignment
Route tickets to appropriate teams based on issue type.

**Requirements:**
1. Use LLM to classify issue category:
   - Hardware, Software, Network, Security, Access Management
2. Map categories to teams:
   - Hardware → Desktop Support Team
   - Software → Application Support Team
   - Network → Network Operations Team
   - Security → Security Team
   - Access Management → Identity Team
3. Add `assigned_team` field to ticket data
4. Update CSV to include team assignment

**Bonus Challenge:** Combine all three parts into one enhanced `process_support_request_v2()` function!

---




## Additional Resources

- **OpenAI Documentation**: https://platform.openai.com/docs
- **Pydantic Docs**: https://docs.pydantic.dev
- **Pandas Guide**: https://pandas.pydata.org/docs


